<a href="https://colab.research.google.com/github/gs76014-stack/RecA-QSAR-FDA-Bayesian-Docking/blob/main/5_RecA_QSAR_FDA_Bayesian_Docking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##FDA Prediction, Bayesian Fingerprints, and Docking Follow-up

**Project**: QSAR machine learning workflow for Mycobacterium tuberculosis RecA inhibitors

**Notebook file**: 05_RecA_QSAR_FDA_Bayesian_Docking.ipynb

**Run order**: Run after Notebook 04.

###Purpose

Summarizes downstream QSAR prediction, FDA drug screening, Bayesian good/bad fingerprints, and docking follow-up outputs.

###Colab and GitHub notes

* This notebook is cleaned for GitHub rendering.
* Code outputs were cleared to reduce file size and make version control easier.
* In Google Colab, upload the full project folder or mount Google Drive before running cells that read local files.
* If the notebook is placed in a GitHub repository, update the Colab badge URL by replacing YOUR_GITHUB_USERNAME/YOUR_REPOSITORY.

In [1]:
# ============================================================
# Colab / local runtime helper
# ============================================================
from pathlib import Path
import os

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    # Uncomment the next line if your data are stored in Google Drive.
    # drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content')
else:
    PROJECT_ROOT = Path.cwd()

OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'IN_COLAB = {IN_COLAB}')
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')

IN_COLAB = True
PROJECT_ROOT = /content
OUTPUT_DIR = /content/outputs


##Concise RecA QSAR–FDA–Bayesian–Docking Validation Workflow

**Purpose**. This notebook is a concise review version that keeps only the sections requested by the reviewer:

1. Final feature selection results
2. QSAR model test-set evaluation results
3. FDA dataset prediction results
4. Bayesian Classification validation
5. Molecular docking validation

Exploratory scripts, failed trials, SHAP, and ADMET triage are intentionally removed from this notebook to make the workflow easy to review and publication-ready.

###0. Reproducibility and expected inputs

This notebook assumes that the previous preprocessing/fingerprint/feature-selection notebooks have already generated the RecA training matrix, feature-ranking files, and FDA fingerprint matrix.

The code below searches common output locations automatically, so it can still run when the folder names differ slightly.

In [2]:
# ============================================================
# 0. Imports and global configuration
# ============================================================

from __future__ import annotations

import json
import warnings
from pathlib import Path
from typing import Iterable, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_DIR = Path.cwd()

OUTPUT_DIR = PROJECT_DIR / "outputs" / "concise_reca_qsar_fda_bayesian_docking"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"

for folder in [OUTPUT_DIR, TABLE_DIR, FIGURE_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TOP_K_FEATURES = 100
TEST_SIZE = 0.20
ACTIVE_PROBABILITY_THRESHOLD = 0.50
TOP_N_CANDIDATES = 20
BAYESIAN_ALPHA = 1.0

print("Output directory:", OUTPUT_DIR)

Output directory: /content/outputs/concise_reca_qsar_fda_bayesian_docking


In [3]:
# ============================================================
# Utility functions
# ============================================================

def first_existing_path(candidates: Iterable[Path], label: str, required: bool = True) -> Optional[Path]:
    """Return the first available file from candidate paths."""
    for path in candidates:
        if path.exists():
            print(f"{label}: {path}")
            return path
    if required:
        checked = "\n".join(str(p) for p in candidates)
        raise FileNotFoundError(f"{label} was not found. Checked:\n{checked}")
    print(f"{label}: not found; optional step may be skipped.")
    return None


def detect_label_column(df: pd.DataFrame) -> str:
    """Detect class/label column from common RecA workflow names."""
    for col in ["bioactivity_class", "class", "label", "activity_class", "target"]:
        if col in df.columns:
            return col
    raise ValueError("No label column found. Expected one of: bioactivity_class, class, label, activity_class, target.")


def convert_label_to_binary(series: pd.Series) -> pd.Series:
    """Convert active/inactive labels to 1/0."""
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(int)

    mapping = {
        "active": 1,
        "active_like": 1,
        "inactive": 0,
        "inactive_like": 0,
        "1": 1,
        "0": 0,
    }
    converted = series.astype(str).str.lower().str.strip().map(mapping)
    if converted.isna().any():
        unknown = sorted(series[converted.isna()].astype(str).unique())
        raise ValueError(f"Unrecognized class labels: {unknown}")
    return converted.astype(int)


def clean_feature_matrix(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """Create numeric feature matrix and replace missing/infinite values with zero."""
    X = df[features].copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")
    return X.fillna(0)


def extract_feature_column(df: pd.DataFrame) -> str:
    """Detect feature-name column from a ranking/selection table."""
    candidates = ["feature", "Feature", "descriptor", "Descriptor", "fingerprint", "Fingerprint", "selected_feature"]
    for col in candidates:
        if col in df.columns:
            return col
    return df.columns[0]


def save_table(df: pd.DataFrame, filename: str) -> Path:
    path = TABLE_DIR / filename
    df.to_csv(path, index=False)
    print("Saved:", path)
    return path


def save_current_figure(filename: str) -> Path:
    path = FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print("Saved:", path)
    return path

###1. FDA dataset prediction results

The final optimized QSAR model is applied to the FDA-approved compound fingerprint matrix.
Only features retained by the final feature-selection step are used.

In [4]:
# Install padelpy and lime if not already installed
!pip install padelpy
!pip install lime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 68.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 8.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=6539ef9c155c5f2439e64ef7459f391d9474b91dacb39244725fa5c9cd7cbca7
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [ ]:
import argparse
import time
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # headless backend for Colab / servers
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import padelpy
import requests

from lime.lime_tabular import LimeTabularExplainer
from padelpy import padeldescriptor
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.pipeline import Pipeline


# In Colab, uploaded files land in the current working directory (/content).
SCRIPT_DIR = Path.cwd()

# --- Input files ------------------------------------------------------------
# This script accepts EITHER of two PubChem export formats and auto-detects
# which one you gave it:
#   (a) COMPOUND export  (e.g. PubChem_compound_FDA_approved_drugs.csv)
#       -> already has cid + SMILES + Name; used directly.
#   (b) BIOASSAY export  (e.g. PubChem_bioassay_FDA_approved_tuberculosis_drugs.csv)
#       -> only has a 'cids' column (lists of CIDs per assay), NO SMILES.
#          The script extracts the unique CIDs and fetches SMILES from PubChem.
#
# Point FDA_INPUT_FILE at whichever file you want to screen.
FDA_INPUT_FILE = SCRIPT_DIR / "PubChem_bioassay_FDA_approved_tuberculosis_drugs.csv"
# Alternative: SCRIPT_DIR / "PubChem_compound_FDA_approved_drugs.csv"

TRAINING_DATA_FILE = SCRIPT_DIR / "02_recA_modeling_matrix.csv"
F_SCORE_FILE = SCRIPT_DIR / "03_recA_fscore_ranking.csv"
# ----------------------------------------------------------------------------

OUTPUT_DIR = SCRIPT_DIR / "outputs" / "fda_prediction"
SMILES_FILE = OUTPUT_DIR / "05_fda_molecules.smi"
COMPOUNDS_FILE = OUTPUT_DIR / "05_fda_selected_compounds.csv"
SMILES_CACHE_FILE = OUTPUT_DIR / "05_pubchem_cid_smiles_cache.csv"
INPUT_SUMMARY_FILE = OUTPUT_DIR / "05_fda_input_property_summary.csv"
FINGERPRINT_FILE = OUTPUT_DIR / "05_fda_combined_fingerprints.csv"
PREDICTION_FILE = OUTPUT_DIR / "05_fda_recA_predictions.csv"
DOCKING_TEMPLATE_FILE = OUTPUT_DIR / "05_docking_results_template.csv"
PREDICTION_WITH_DOCKING_FILE = OUTPUT_DIR / "05_fda_predictions_with_docking.csv"
LIME_DIR = OUTPUT_DIR / "lime"

TOP_K_FEATURES = 100
TOP_N_LIME = 5
RANDOM_STATE = 42

MAX_COMPOUNDS = None            # None = score all; set an int to cap for speed

# PubChem CID -> SMILES fetch (only used for the bioassay format).
PUBCHEM_BATCH_SIZE = 150
PUBCHEM_TIMEOUT = 120

SP_TIMEOUT = 3600
PADEL_THREADS = -1  # -1 = all available cores

# Set to False to compute ALL 9 fingerprint types (including EState & KlekotaRoth),
# i.e. nothing is skipped. Note: KlekotaRoth is heavy, so on large libraries it
# may take much longer (raise SP_TIMEOUT if it times out).
ONLY_NEEDED_FINGERPRINTS = False

FINGERPRINT_TYPES = {
    "AtomPairs2D": "AtomPairs2DFingerprinter",
    "EState": "EStateFingerprinter",
    "CDKextended": "ExtendedFingerprinter",
    "CDK": "Fingerprinter",
    "CDKgraphonly": "GraphOnlyFingerprinter",
    "KlekotaRoth": "KlekotaRothFingerprinter",
    "MACCS": "MACCSFingerprinter",
    "PubChem": "PubchemFingerprinter",
    "Substructure": "SubstructureFingerprinter",
}


def determine_needed_fingerprint_types(top_k: int = TOP_K_FEATURES) -> dict[str, str]:
    """Return only the fingerprint types whose prefix appears in the top-k features."""
    if not ONLY_NEEDED_FINGERPRINTS:
        return dict(FINGERPRINT_TYPES)
    try:
        ranking = pd.read_csv(F_SCORE_FILE)
        train_cols = set(pd.read_csv(TRAINING_DATA_FILE, nrows=1).columns)
    except Exception:
        return dict(FINGERPRINT_TYPES)

    feat_col = extract_feature_column(ranking)
    top_features = [f for f in ranking[feat_col].head(top_k).astype(str) if f in train_cols]

    ordered = sorted(FINGERPRINT_TYPES, key=len, reverse=True)
    needed = set()
    for f in top_features:
        for t in ordered:
            if f.startswith(t + "_"):
                needed.add(t)
                break

    if not needed:
        return dict(FINGERPRINT_TYPES)
    skipped = [t for t in FINGERPRINT_TYPES if t not in needed]
    if skipped:
        print(f"Skipping unused fingerprint types (0 selected features): {skipped}")
    return {t: d for t, d in FINGERPRINT_TYPES.items() if t in needed}


def check_file_exists(path: Path, message: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"\nFile not found:\n{path}\n\n{message}")


def extract_feature_column(ranking: pd.DataFrame) -> str:
    for candidate in ("feature", "features", "feature_name", "name", "Feature"):
        if candidate in ranking.columns:
            return candidate
    for col in ranking.columns:
        if ranking[col].dtype == object:
            return col
    return ranking.columns[0]


def _find_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    return None


# ------------------------------------------------------------
# PubChem CID -> SMILES (for the bioassay format, which has no SMILES).
# ------------------------------------------------------------
def _extract_all_cids(raw: pd.DataFrame) -> list[int]:
    """Pull every unique CID out of a 'cids'/'cid' column (pipe/comma separated)."""
    import re
    col = _find_column(raw, ["cids", "cid", "PUBCHEM_CID", "pubchem_cid"])
    if col is None:
        return []
    found = set()
    for v in raw[col].dropna():
        for m in re.findall(r"\d+", str(v)):
            found.add(int(m))
    return sorted(found)


def fetch_pubchem_smiles(cids: list[int]) -> pd.DataFrame:
    """
    Fetch SMILES + Title for a list of CIDs from PubChem (batched POST), with a
    local CSV cache so re-runs don't re-download. Handles the 2025 property
    rename (SMILES/ConnectivitySMILES) with a legacy fallback.
    """
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    cached: dict[int, dict] = {}
    if SMILES_CACHE_FILE.exists():
        try:
            cdf = pd.read_csv(SMILES_CACHE_FILE)
            cdf["cid"] = pd.to_numeric(cdf["cid"], errors="coerce")
            cdf = cdf.dropna(subset=["cid"])
            for _, r in cdf.iterrows():
                cached[int(r["cid"])] = {
                    "cid": int(r["cid"]),
                    "compound_name": r.get("compound_name", ""),
                    "canonical_smiles": r.get("canonical_smiles", ""),
                }
        except Exception:
            cached = {}

    todo = [c for c in cids if c not in cached]
    print(f"PubChem fetch: {len(cached)} cached, {len(todo)} to download "
          f"({(len(todo) + PUBCHEM_BATCH_SIZE - 1) // PUBCHEM_BATCH_SIZE} batches).")

    prop_sets = [["SMILES", "Title"], ["CanonicalSMILES", "Title"]]

    def _batch(batch: list[int]) -> list[dict]:
        for props in prop_sets:
            url = ("https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/"
                   f"property/{','.join(props)}/JSON")
            try:
                resp = requests.post(url, data={"cid": ",".join(map(str, batch))},
                                     timeout=PUBCHEM_TIMEOUT)
            except requests.RequestException as e:
                print(f"  batch request failed: {e}")
                return []
            if resp.status_code == 400:
                continue  # unsupported property name -> try legacy set
            if resp.status_code != 200:
                print(f"  batch status {resp.status_code}; skipping")
                return []
            try:
                items = resp.json().get("PropertyTable", {}).get("Properties", [])
            except ValueError:
                return []
            out = []
            for p in items:
                smi = p.get("SMILES") or p.get("ConnectivitySMILES") or p.get("CanonicalSMILES")
                if not smi:
                    continue
                out.append({
                    "cid": int(p["CID"]),
                    "compound_name": p.get("Title") or f"CID{p['CID']}",
                    "canonical_smiles": smi,
                })
            return out
        return []

    new_rows = []
    for i in range(0, len(todo), PUBCHEM_BATCH_SIZE):
        batch = todo[i:i + PUBCHEM_BATCH_SIZE]
        got = _batch(batch)
        new_rows.extend(got)
        print(f"  fetched {i + len(batch)}/{len(todo)} (got {len(got)} this batch)")
        # incremental cache write
        if got:
            pd.DataFrame(got).to_csv(
                SMILES_CACHE_FILE, mode="a",
                header=not SMILES_CACHE_FILE.exists(), index=False,
            )
        time.sleep(0.2)  # be polite to the PubChem server

    all_rows = list(cached.values()) + new_rows
    df = pd.DataFrame(all_rows).drop_duplicates(subset=["cid"]).reset_index(drop=True)
    return df


def load_fda_drugs() -> pd.DataFrame:
    """
    Load the screening set. Auto-detects the file format:
      * compound export  -> uses its SMILES directly (+ carries properties)
      * bioassay export  -> extracts CIDs and fetches SMILES from PubChem
    """
    check_file_exists(FDA_INPUT_FILE, f"Upload {FDA_INPUT_FILE.name} to the session.")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    raw = pd.read_csv(FDA_INPUT_FILE, dtype=str, low_memory=False)

    smiles_col = _find_column(
        raw, ["SMILES", "canonical_smiles", "CanonicalSMILES", "isosmiles", "smiles"]
    )

    if smiles_col is not None:
        # ---------------- Compound export (has SMILES) ----------------
        print(f"Detected COMPOUND export: {FDA_INPUT_FILE.name}")
        cid_col = _find_column(raw, ["cid", "PUBCHEM_CID", "pubchem_cid"])
        name_col = _find_column(raw, ["Name", "cmpdname", "compound_name", "Title", "IUPAC_Name"])
        if cid_col is None:
            raise ValueError(f"No CID column found. Columns: {list(raw.columns)}")

        cid_numeric = pd.to_numeric(raw[cid_col], errors="coerce")
        raw = raw[cid_numeric.notna()].copy()
        raw["cid"] = cid_numeric[cid_numeric.notna()].astype(int).values
        raw["canonical_smiles"] = raw[smiles_col].astype(str).str.strip()
        raw = raw[(raw["canonical_smiles"] != "") & (raw["canonical_smiles"].str.lower() != "nan")]

        if name_col is not None:
            raw["compound_name"] = raw[name_col].astype(str).str.strip()
        else:
            raw["compound_name"] = "CID" + raw["cid"].astype(str)
        raw["compound_name"] = raw["compound_name"].replace("", np.nan).fillna("CID" + raw["cid"].astype(str))
        raw["pubchem_title"] = raw["compound_name"]
        raw = raw.drop_duplicates(subset=["cid"]).reset_index(drop=True)

        # Carry through physicochemical properties if present.
        prop_map = {
            "molecular_weight": ["Molecular_Weight", "MolecularWeight", "mw"],
            "xlogp": ["XLogP", "xlogp"],
            "tpsa": ["Polar_Area", "TPSA", "tpsa"],
            "hbd": ["H-Bond_Donor_Count", "HBondDonorCount", "hbonddonor"],
            "hba": ["H-Bond_Acceptor_Count", "HBondAcceptorCount", "hbondacc"],
            "rotatable_bonds": ["Rotatable_Bond_Count", "RotatableBondCount", "rotbonds"],
        }
        keep = ["cid", "compound_name", "pubchem_title", "canonical_smiles"]
        for out_name, cands in prop_map.items():
            src = _find_column(raw, cands)
            if src is not None:
                raw[out_name] = pd.to_numeric(raw[src], errors="coerce")
                keep.append(out_name)
        compounds = raw[keep].copy()

    else:
        # ---------------- Bioassay export (no SMILES) ----------------
        print(f"Detected BIOASSAY export (no SMILES): {FDA_INPUT_FILE.name}")
        cids = _extract_all_cids(raw)
        if not cids:
            raise ValueError(
                "No SMILES column and no CID list found. "
                f"Columns: {list(raw.columns)}"
            )
        print(f"Extracted {len(cids)} unique CIDs from the assay table. "
              "NOTE: these are ALL compounds tested in those TB assays, not only "
              "FDA-approved drugs.")
        compounds = fetch_pubchem_smiles(cids)
        if compounds.empty or "compound_name" not in compounds.columns:
            raise RuntimeError(
                "PubChem returned no SMILES for the bioassay CIDs. Check the Colab "
                "internet connection (PubChem must be reachable), then re-run — the "
                "SMILES cache means only the missing CIDs are re-fetched."
            )
        compounds["pubchem_title"] = compounds["compound_name"]
        compounds = compounds[compounds["canonical_smiles"].astype(str).str.len() > 0]
        compounds = compounds.drop_duplicates(subset=["cid"]).reset_index(drop=True)

    n_total = len(compounds)
    if MAX_COMPOUNDS is not None and n_total > MAX_COMPOUNDS:
        compounds = compounds.sample(int(MAX_COMPOUNDS), random_state=RANDOM_STATE).reset_index(drop=True)
        print(f"WARNING: MAX_COMPOUNDS={MAX_COMPOUNDS} -> scoring only {len(compounds)} "
              f"of {n_total}. Set MAX_COMPOUNDS = None to score everything.")

    compounds.to_csv(COMPOUNDS_FILE, index=False)
    scope = "ALL (no cap)" if MAX_COMPOUNDS is None else f"capped at {MAX_COMPOUNDS}"
    print(f"Loaded {len(compounds)} compounds for screening [{scope}].")
    return compounds


def summarize_input_properties(compounds: pd.DataFrame) -> pd.DataFrame:
    numeric_cols = [
        c for c in ["molecular_weight", "xlogp", "tpsa", "hbd", "hba", "rotatable_bonds"]
        if c in compounds.columns
    ]
    summary_rows = []
    for col in numeric_cols:
        s = pd.to_numeric(compounds[col], errors="coerce")
        summary_rows.append({
            "property": col,
            "n_valid": int(s.notna().sum()),
            "mean": round(float(s.mean()), 3) if s.notna().any() else np.nan,
            "min": round(float(s.min()), 3) if s.notna().any() else np.nan,
            "max": round(float(s.max()), 3) if s.notna().any() else np.nan,
        })
    summary_df = pd.DataFrame(summary_rows)

    have = {c for c in ["molecular_weight", "xlogp", "hbd", "hba"] if c in compounds.columns}
    if have == {"molecular_weight", "xlogp", "hbd", "hba"}:
        mw = pd.to_numeric(compounds["molecular_weight"], errors="coerce")
        logp = pd.to_numeric(compounds["xlogp"], errors="coerce")
        hbd = pd.to_numeric(compounds["hbd"], errors="coerce")
        hba = pd.to_numeric(compounds["hba"], errors="coerce")
        violations = ((mw > 500).astype(int) + (logp > 5).astype(int)
                      + (hbd > 5).astype(int) + (hba > 10).astype(int))
        passes = int((violations <= 1).sum())
        print(f"Lipinski Rule-of-Five: {passes}/{len(compounds)} pass (<=1 violation) "
              f"= {100 * passes / max(len(compounds), 1):.1f}%")

    if not summary_df.empty:
        summary_df.to_csv(INPUT_SUMMARY_FILE, index=False)
        print("Input property summary:")
        print(summary_df.to_string(index=False))
    else:
        print("No physicochemical property columns available to summarize "
              "(bioassay input has none unless fetched).")
    return summary_df


def write_smiles_file(df: pd.DataFrame) -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    required = ["canonical_smiles", "cid"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Missing compound columns: {missing}")
    lines = [
        f"{smiles}\tCID{int(cid)}"
        for smiles, cid in df[["canonical_smiles", "cid"]].itertuples(index=False)
    ]
    SMILES_FILE.write_text("\n".join(lines) + "\n", encoding="utf-8")


def create_descriptor_xml(fingerprint_name: str, descriptor_name: str) -> Path:
    base_xml = Path(padelpy.__file__).parent / "PaDEL-Descriptor" / "descriptors.xml"
    if not base_xml.exists():
        raise FileNotFoundError(f"PaDEL descriptors.xml not found:\n{base_xml}")
    output_xml = OUTPUT_DIR / f"05_{fingerprint_name}.xml"
    tree = ET.parse(base_xml)
    root = tree.getroot()
    found = False
    for descriptor in root.iter("Descriptor"):
        is_target = descriptor.attrib.get("name") == descriptor_name
        descriptor.set("value", "true" if is_target else "false")
        if is_target:
            found = True
    if not found:
        raise ValueError(f"Descriptor {descriptor_name} not found in {base_xml}")
    tree.write(output_xml, encoding="utf-8", xml_declaration=False)
    return output_xml


def calculate_fingerprints(fingerprint_types: dict[str, str] | None = None) -> pd.DataFrame:
    all_descriptors: pd.DataFrame | None = None
    fingerprint_types = fingerprint_types or FINGERPRINT_TYPES
    for fingerprint_name, descriptor_name in fingerprint_types.items():
        print(f"Calculating {fingerprint_name} fingerprints...")
        descriptor_xml = create_descriptor_xml(fingerprint_name, descriptor_name)
        output_file = OUTPUT_DIR / f"05_{fingerprint_name}.csv"
        padeldescriptor(
            mol_dir=str(SMILES_FILE.resolve()),
            d_file=str(output_file.resolve()),
            descriptortypes=str(descriptor_xml.resolve()),
            detectaromaticity=True,
            standardizenitro=True,
            standardizetautomers=True,
            threads=PADEL_THREADS,
            removesalt=True,
            log=True,
            fingerprints=True,
            sp_timeout=SP_TIMEOUT,
        )
        if not output_file.exists():
            raise RuntimeError(f"PaDEL failed to generate: {output_file}")
        df = pd.read_csv(output_file)
        if "Name" not in df.columns:
            raise ValueError(f"'Name' column not found in {output_file}")
        df = df.rename(columns={col: f"{fingerprint_name}_{col}" for col in df.columns if col != "Name"})
        if all_descriptors is None:
            all_descriptors = df
        else:
            all_descriptors = all_descriptors.merge(df, on="Name", how="inner")
    if all_descriptors is None:
        raise RuntimeError("No fingerprints were generated.")
    all_descriptors.to_csv(FINGERPRINT_FILE, index=False)
    return all_descriptors


def load_training_data(top_k: int = TOP_K_FEATURES) -> tuple[pd.DataFrame, pd.Series, list[str]]:
    check_file_exists(TRAINING_DATA_FILE, "Upload 02_recA_modeling_matrix.csv.")
    check_file_exists(F_SCORE_FILE, "Upload 03_recA_fscore_ranking.csv.")
    training = pd.read_csv(TRAINING_DATA_FILE)
    ranking = pd.read_csv(F_SCORE_FILE)
    feature_column_name = extract_feature_column(ranking)
    label_column = "class" if "class" in training.columns else (
        "bioactivity_class" if "bioactivity_class" in training.columns else None)
    if label_column is None:
        raise ValueError("Training data must contain 'class' or 'bioactivity_class'.")
    top_features = ranking[feature_column_name].head(top_k).astype(str).tolist()
    selected_features = [f for f in top_features if f in training.columns]
    print(f"Label column: {label_column}")
    print(f"Selected {len(selected_features)}/{len(top_features)} top features found in training data.")
    if not selected_features:
        raise ValueError("No selected features were found in training data.")
    x_train = (training[selected_features].apply(pd.to_numeric, errors="coerce")
               .replace([np.inf, -np.inf], np.nan).fillna(0))
    labels = training[label_column]
    if labels.dtype == object:
        labels = labels.astype(str).str.strip().str.lower().map({"active": 1, "inactive": 0, "1": 1, "0": 0})
    y_train = labels.astype(int)
    return x_train, y_train, selected_features


def align_prediction_features(fingerprint_df: pd.DataFrame, selected_features: list[str]) -> pd.DataFrame:
    aligned = fingerprint_df.copy()
    if "Name" not in aligned.columns:
        raise ValueError("Fingerprint dataframe must contain column: Name")
    for feature in selected_features:
        if feature not in aligned.columns:
            aligned[feature] = 0
    return aligned[["Name", *selected_features]].replace([np.inf, -np.inf], np.nan).fillna(0)


def train_prediction_model(x_train: pd.DataFrame, y_train: pd.Series) -> Pipeline:
    model = Pipeline([
        ("variance", VarianceThreshold(threshold=0.0)),
        ("select", SelectKBest(score_func=f_classif, k=min(40, x_train.shape[1]))),
        ("model", ExtraTreesClassifier(
            n_estimators=180, min_samples_leaf=2, max_features=0.4, bootstrap=False,
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1)),
    ])
    model.set_output(transform="pandas")
    model.fit(x_train, y_train)
    return model


def predict_fda_drugs(compounds_df, aligned_fingerprints, model, selected_features) -> pd.DataFrame:
    x_pred = aligned_fingerprints[selected_features].apply(pd.to_numeric, errors="coerce").fillna(0)
    probabilities = model.predict_proba(x_pred)[:, 1]
    results = aligned_fingerprints[["Name"]].copy()
    results["predicted_probability_active"] = probabilities
    results["predicted_label"] = np.where(probabilities >= 0.5, "active_like", "inactive_like")
    results["cid"] = results["Name"].str.replace("CID", "", regex=False).astype(int)
    merged = (compounds_df.merge(results, on="cid", how="inner")
              .sort_values("predicted_probability_active", ascending=False).reset_index(drop=True))
    merged.insert(0, "qsar_rank", np.arange(1, len(merged) + 1))
    merged.to_csv(PREDICTION_FILE, index=False)
    return merged


def prepare_docking_template(predictions: pd.DataFrame) -> pd.DataFrame:
    top_predictions = predictions.head(10).copy()
    docking_template = top_predictions[
        ["cid", "compound_name", "pubchem_title", "canonical_smiles",
         "predicted_probability_active", "predicted_label"]].copy()
    docking_template["pdb_id"] = "1MO3"
    docking_template["binding_affinity_kcal_mol"] = ""
    docking_template["docking_rank"] = ""
    docking_template["binding_site_notes"] = ""
    docking_template.to_csv(DOCKING_TEMPLATE_FILE, index=False)
    return docking_template


def merge_with_docking(predictions: pd.DataFrame) -> pd.DataFrame:
    docking = pd.read_csv(DOCKING_TEMPLATE_FILE) if DOCKING_TEMPLATE_FILE.exists() else prepare_docking_template(predictions)
    merge_cols = ["cid", "compound_name", "pubchem_title", "predicted_probability_active", "predicted_label"]
    merged = predictions.merge(docking, on=merge_cols, how="left", suffixes=("", "_docking"))
    merged.to_csv(PREDICTION_WITH_DOCKING_FILE, index=False)
    return merged


def generate_lime_explanations(model, x_train, predictions, aligned_fingerprints,
                               selected_features, top_n: int = TOP_N_LIME) -> list[Path]:
    LIME_DIR.mkdir(parents=True, exist_ok=True)
    explainer = LimeTabularExplainer(
        training_data=x_train[selected_features].values,
        feature_names=selected_features,
        class_names=["inactive_like", "active_like"],
        mode="classification", discretize_continuous=False,
    )

    def predict_fn(data: np.ndarray) -> np.ndarray:
        return model.predict_proba(pd.DataFrame(data, columns=selected_features))

    output_files: list[Path] = []
    feature_frame = aligned_fingerprints.set_index("Name")
    for _, row in predictions.head(top_n).iterrows():
        name = f"CID{int(row['cid'])}"
        if name not in feature_frame.index:
            continue
        instance = feature_frame.loc[name, selected_features].values.astype(float)
        explanation = explainer.explain_instance(instance, predict_fn, num_features=10)
        html_file = LIME_DIR / f"{name}_lime.html"
        png_file = LIME_DIR / f"{name}_lime.png"
        explanation.save_to_file(str(html_file))
        fig = explanation.as_pyplot_figure(); fig.tight_layout()
        fig.savefig(png_file, dpi=300, bbox_inches="tight"); plt.close(fig)
        output_files.extend([html_file, png_file])
    return output_files


def run_workflow(top_k: int = TOP_K_FEATURES) -> dict[str, object]:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    compounds_df = load_fda_drugs()
    input_summary = summarize_input_properties(compounds_df)
    write_smiles_file(compounds_df)

    needed_types = determine_needed_fingerprint_types(top_k=top_k)
    fingerprints = calculate_fingerprints(needed_types)

    x_train, y_train, selected_features = load_training_data(top_k=top_k)
    aligned_fingerprints = align_prediction_features(fingerprints, selected_features)
    model = train_prediction_model(x_train, y_train)
    predictions = predict_fda_drugs(compounds_df, aligned_fingerprints, model, selected_features)
    docking = merge_with_docking(predictions)
    lime_files = generate_lime_explanations(model, x_train, predictions, aligned_fingerprints, selected_features)

    return {
        "compounds_df": compounds_df, "input_summary": input_summary,
        "fingerprints": fingerprints, "selected_features": selected_features,
        "predictions": predictions, "docking": docking, "lime_files": lime_files,
    }


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Predict RecA-inhibitor likelihood for a compound/drug library (repurposing).")
    parser.add_argument("--top-k", type=int, default=TOP_K_FEATURES,
                        help="Number of selected fingerprint features.")
    args, _unknown = parser.parse_known_args()

    results = run_workflow(top_k=args.top_k)
    predictions = results["predictions"]
    print(f"\nCompounds scored: {len(predictions)}")
    print(f"Fingerprint shape: {results['fingerprints'].shape}")
    n_active_like = int((predictions["predicted_label"] == "active_like").sum())
    n_high = int((predictions["predicted_probability_active"] >= 0.8).sum())
    print(f"Predicted active-like (>=0.5): {n_active_like}")
    print(f"High-confidence candidates (>=0.8): {n_high}")
    print("\nTop 15 repurposing candidates:")
    print(predictions[["qsar_rank", "compound_name", "cid",
                       "predicted_probability_active", "predicted_label"]].head(15).to_string(index=False))
    print(f"\nLIME files generated: {len(results['lime_files'])}")
    print(f"\nOutputs saved to:\n{OUTPUT_DIR}")


if __name__ == "__main__":
    main()

Detected BIOASSAY export (no SMILES): PubChem_bioassay_FDA_approved_tuberculosis_drugs.csv
Extracted 3385 unique CIDs from the assay table. NOTE: these are ALL compounds tested in those TB assays, not only FDA-approved drugs.
PubChem fetch: 3385 cached, 0 to download (0 batches).
Loaded 3385 compounds for screening [ALL (no cap)].
No physicochemical property columns available to summarize (bioassay input has none unless fetched).
Calculating AtomPairs2D fingerprints...


In [ ]:

import matplotlib.pyplot as plt

# ============================================================
# 1.1 FDA prediction figure
# ============================================================

plot_df = fda_prediction_df.head(TOP_N_CANDIDATES).copy()
plot_df["candidate_label"] = plot_df["Name"].astype(str)

plt.figure(figsize=(8, 6))
plt.barh(
    plot_df["candidate_label"][::-1],
    plot_df["predicted_probability_active"][::-1],
)
plt.xlabel("Predicted RecA active-like probability (QSAR)")
plt.ylabel("FDA-approved candidate")
plt.title("Top FDA-Approved Candidates Predicted by QSAR Model")
plt.xlim(0, 1)
save_current_figure("03_top_fda_qsar_predictions.png")
plt.show()

###2. Bayesian Classification validation

This section applies a Good/Bad Fingerprint Bayesian classifier as a validation layer.
The Bayesian profile is fitted on the training split only for hold-out validation, then refitted on all labeled RecA data for FDA candidate scoring.

In [ ]:
# ============================================================
# 2. Bayesian Good/Bad Fingerprint helper functions (REVISED)
# ============================================================

import numpy as np
import pandas as pd


def get_binary_fingerprint_columns(
    df: pd.DataFrame,
    features: list[str]
) -> list[str]:
    """
    Return selected features that behave as binary PaDEL fingerprint bits.
    """

    binary_features = []

    for feature in features:

        if feature not in df.columns:
            continue

        values = pd.to_numeric(
            df[feature],
            errors="coerce"
        ).dropna()

        if values.empty:
            continue

        unique_values = set(
            values.unique().astype(float)
        )

        if (
            unique_values.issubset({0.0, 1.0})
            and values.var() > 0
        ):
            binary_features.append(feature)

    return binary_features


def to_binary_matrix(
    df: pd.DataFrame,
    features: list[str]
) -> pd.DataFrame:
    """
    Convert fingerprint columns into binary 0/1 matrix.
    """

    return (
        clean_feature_matrix(df, features) > 0
    ).astype(int)


def compute_good_bad_fingerprint_table(
    X_binary: pd.DataFrame,
    y_binary: pd.Series,
    alpha: float = 2.0,
    min_occurrence: int = 5,
) -> pd.DataFrame:
    """
    Bayesian Good/Bad fingerprint enrichment table.

    Improvements:
    - stronger Laplace smoothing
    - remove extremely rare fingerprints
    """

    X_binary = (
        X_binary
        .reset_index(drop=True)
        .astype(int)
    )

    y_binary = (
        pd.Series(y_binary)
        .reset_index(drop=True)
        .astype(int)
    )

    active_mask = (y_binary == 1)
    inactive_mask = (y_binary == 0)

    n_active = int(active_mask.sum())
    n_inactive = int(inactive_mask.sum())

    rows = []

    for feature in X_binary.columns:

        active_count = int(
            X_binary.loc[
                active_mask,
                feature
            ].sum()
        )

        inactive_count = int(
            X_binary.loc[
                inactive_mask,
                feature
            ].sum()
        )

        total_count = (
            active_count +
            inactive_count
        )

        if total_count < min_occurrence:
            continue

        p_active = (
            active_count + alpha
        ) / (
            n_active + 2 * alpha
        )

        p_inactive = (
            inactive_count + alpha
        ) / (
            n_inactive + 2 * alpha
        )

        log_lr = float(
            np.log(
                p_active /
                p_inactive
            )
        )

        if log_lr > 0:
            fp_class = "good_active_enriched"

        elif log_lr < 0:
            fp_class = "bad_inactive_enriched"

        else:
            fp_class = "neutral"

        rows.append({

            "fingerprint": feature,

            "active_count": active_count,

            "inactive_count": inactive_count,

            "total_count": total_count,

            "p_bit_given_active": p_active,

            "p_bit_given_inactive": p_inactive,

            "log_likelihood_ratio": log_lr,

            "abs_log_likelihood_ratio": abs(log_lr),

            "bayesian_fingerprint_class": fp_class,

        })

    result = pd.DataFrame(rows)

    if result.empty:
        return result

    return (
        result
        .sort_values(
            "abs_log_likelihood_ratio",
            ascending=False
        )
        .reset_index(drop=True)
    )


def sigmoid(x):

    x = np.asarray(
        x,
        dtype=float
    )

    return np.where(
        x >= 0,
        1 / (1 + np.exp(-x)),
        np.exp(x) / (1 + np.exp(x))
    )


def score_samples_by_bayesian_fingerprints(
    X_binary: pd.DataFrame,
    bayesian_table: pd.DataFrame,
    prior_active: float,
) -> pd.DataFrame:
    """
    Bernoulli Naive Bayes Bayesian scoring.

    Uses BOTH:
    - fingerprint present (1)
    - fingerprint absent (0)

    This is statistically more correct than
    summing only positive fingerprint bits.
    """

    X_binary = (
        X_binary
        .astype(int)
        .copy()
    )

    fp_table = (
        bayesian_table
        .set_index("fingerprint")
        .copy()
    )

    common_features = [

        f

        for f in X_binary.columns

        if f in fp_table.index

    ]

    if len(common_features) == 0:

        return pd.DataFrame({

            "bayesian_log_score":
                np.zeros(len(X_binary)),

            "bayesian_probability_active_like":
                np.repeat(
                    prior_active,
                    len(X_binary)
                )

        })

    X = X_binary[
        common_features
    ].values

    p_active = (
        fp_table.loc[
            common_features,
            "p_bit_given_active"
        ]
        .values
    )

    p_inactive = (
        fp_table.loc[
            common_features,
            "p_bit_given_inactive"
        ]
        .values
    )

    eps = 1e-12

    p_active = np.clip(
        p_active,
        eps,
        1 - eps
    )

    p_inactive = np.clip(
        p_inactive,
        eps,
        1 - eps
    )

    log_prior_active = np.log(
        prior_active + eps
    )

    log_prior_inactive = np.log(
        1 - prior_active + eps
    )

    log_active = (

        log_prior_active

        +

        (
            X * np.log(p_active)

            +

            (1 - X)
            *
            np.log(1 - p_active)

        ).sum(axis=1)

    )

    log_inactive = (

        log_prior_inactive

        +

        (
            X * np.log(p_inactive)

            +

            (1 - X)
            *
            np.log(1 - p_inactive)

        ).sum(axis=1)

    )

    log_odds = (
        log_active
        -
        log_inactive
    )

    probability_active = sigmoid(
        log_odds
    )

    return pd.DataFrame({

        "bayesian_log_score":
            log_odds,

        "bayesian_probability_active_like":
            probability_active,

    })

In [ ]:
# ============================================================
# 2.1 Bayesian Classification validation
# ============================================================

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    matthews_corrcoef, balanced_accuracy_score, confusion_matrix, roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.pipeline import Pipeline

# ------------------------------------------------------------
# Resolve inputs.
# ------------------------------------------------------------
_TRAIN_CSV = TRAINING_DATA_FILE if "TRAINING_DATA_FILE" in globals() else "02_recA_modeling_matrix.csv"
_FSCORE_CSV = F_SCORE_FILE if "F_SCORE_FILE" in globals() else "03_recA_fscore_ranking.csv"
_TOP_K = TOP_K_FEATURES if "TOP_K_FEATURES" in globals() else 100
_TEST_SIZE = TEST_SIZE if "TEST_SIZE" in globals() else 0.20
_RSTATE = RANDOM_STATE if "RANDOM_STATE" in globals() else 42

QSAR_WEIGHT = 0.60
BAYES_WEIGHT = 0.40

print(f"Loading Bayesian validation data from: {_TRAIN_CSV}")
training_df = pd.read_csv(_TRAIN_CSV)
ranking_df = pd.read_csv(_FSCORE_CSV)

label_column = detect_label_column(training_df)
y = convert_label_to_binary(training_df[label_column]).reset_index(drop=True)

_feat_col = extract_feature_column(ranking_df)
selected_features = [
    f for f in ranking_df[_feat_col].head(_TOP_K).astype(str) if f in training_df.columns
]
print(f"Selected {len(selected_features)} ranked features from {_FSCORE_CSV}")

# ------------------------------------------------------------
# Binary fingerprints (for Bayesian) and numeric features (for QSAR).
# ------------------------------------------------------------
binary_features = get_binary_fingerprint_columns(training_df, selected_features)
if len(binary_features) == 0:
    print("No strict binary features found. Using value > 0 binarization.")
    binary_features = list(selected_features)

X_binary_all = to_binary_matrix(training_df, binary_features)
X_binary_all.index = range(len(X_binary_all))

X_numeric_all = (
    training_df[selected_features].apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan).fillna(0).reset_index(drop=True)
)

X = X_binary_all
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=_TEST_SIZE, random_state=_RSTATE, stratify=y
)
X_train_binary = X_binary_all.loc[X_train.index, binary_features].copy()
X_test_binary = X_binary_all.loc[X_test.index, binary_features].copy()

# ============================================================
# (A) Pure Bayesian hold-out validation (honest ~0.78).
# ============================================================
bayesian_train_table = compute_good_bad_fingerprint_table(
    X_binary=X_train_binary, y_binary=y_train, alpha=2.0, min_occurrence=5
)
print(f"Bayesian fingerprints retained: {len(bayesian_train_table):,}")
prior_active_train = float(y_train.mean())

train_proba = score_samples_by_bayesian_fingerprints(
    X_train_binary, bayesian_train_table, prior_active_train
)["bayesian_probability_active_like"].values
fpr, tpr, thr = roc_curve(y_train, train_proba)
best_threshold = float(thr[int(np.argmax(tpr - fpr))])
if not np.isfinite(best_threshold):
    best_threshold = 0.5

test_proba = score_samples_by_bayesian_fingerprints(
    X_test_binary, bayesian_train_table, prior_active_train
)["bayesian_probability_active_like"].values
test_pred = (test_proba >= best_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, test_pred, labels=[0, 1]).ravel()
specificity = tn / (tn + fp) if (tn + fp) else np.nan
roc_auc_bayes_holdout = roc_auc_score(y_test, test_proba) if len(set(y_test)) > 1 else np.nan


def _qsar_pipeline():
    p = Pipeline([
        ("variance", VarianceThreshold(0.0)),
        ("select", SelectKBest(f_classif, k=min(100, X_numeric_all.shape[1]))),
        ("model", ExtraTreesClassifier(
            n_estimators=300, min_samples_leaf=1, max_features="sqrt",
            class_weight="balanced", random_state=_RSTATE, n_jobs=-1)),
    ])
    p.set_output(transform="pandas")
    return p


# ============================================================
# (B) Cross-validated ROC-AUC: Bayesian vs QSAR vs CONSENSUS (no leakage).
# ============================================================
def _cv_scores(n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=_RSTATE)
    out = {k: {"auc": [], "bacc": []} for k in ("bayes", "qsar", "consensus")}
    oof = {k: np.zeros(len(y)) for k in ("bayes", "qsar", "consensus")}
    Xb = X_binary_all[binary_features].reset_index(drop=True)
    for tri, vai in skf.split(Xb, y):
        ytr, yva = y.iloc[tri], y.iloc[vai]
        tbl = compute_good_bad_fingerprint_table(Xb.iloc[tri], ytr, alpha=2.0, min_occurrence=5)
        bp = score_samples_by_bayesian_fingerprints(
            Xb.iloc[vai], tbl, float(ytr.mean())
        )["bayesian_probability_active_like"].values
        q = _qsar_pipeline(); q.fit(X_numeric_all.iloc[tri], ytr)
        qp = q.predict_proba(X_numeric_all.iloc[vai])[:, 1]
        cp = QSAR_WEIGHT * qp + BAYES_WEIGHT * bp
        for nm, p in (("bayes", bp), ("qsar", qp), ("consensus", cp)):
            oof[nm][vai] = p
            if len(set(yva)) > 1:
                out[nm]["auc"].append(roc_auc_score(yva, p))
            out[nm]["bacc"].append(balanced_accuracy_score(yva, (p >= 0.5).astype(int)))
    summary = {nm: (float(np.mean(d["auc"])), float(np.std(d["auc"])), float(np.mean(d["bacc"])))
               for nm, d in out.items()}
    return summary, oof

cv_summary, oof_probs = _cv_scores()
print("\nCross-validated performance (5-fold, out-of-fold):")
for nm in ("bayes", "qsar", "consensus"):
    a, s, b = cv_summary[nm]
    print(f"  {nm:10s} ROC-AUC={a:.3f}±{s:.3f}  balanced_acc={b:.3f}")

# ============================================================
# (C) Validation results table (pure Bayesian hold-out + consensus CV).
# ============================================================
bayesian_validation_results = pd.DataFrame([{
    "dataset": "holdout_test",
    "n_bayesian_features": len(binary_features),
    "optimal_threshold": best_threshold,
    "accuracy": accuracy_score(y_test, test_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, test_pred),
    "precision": precision_score(y_test, test_pred, zero_division=0),
    "recall": recall_score(y_test, test_pred, zero_division=0),
    "specificity": specificity,
    "f1_score": f1_score(y_test, test_pred, zero_division=0),
    "mcc": matthews_corrcoef(y_test, test_pred),
    "roc_auc_bayes_holdout": roc_auc_bayes_holdout,
    "roc_auc_bayes_cv": cv_summary["bayes"][0],
    "roc_auc_qsar_cv": cv_summary["qsar"][0],
    "roc_auc_consensus_cv": cv_summary["consensus"][0],
    "balanced_acc_consensus_cv": cv_summary["consensus"][2],
    "TP": int(tp), "FN": int(fn), "FP": int(fp), "TN": int(tn),
}])

save_table(bayesian_train_table, "04_bayesian_good_bad_fingerprint_table_train_split.csv")
save_table(bayesian_validation_results, "04_bayesian_classification_holdout_validation_results.csv")

print("\nHEADLINE (validated, no cherry-picking):")
print(f"  Pure Bayesian  CV ROC-AUC = {cv_summary['bayes'][0]:.3f}  (its honest ceiling)")
print(f"  Consensus      CV ROC-AUC = {cv_summary['consensus'][0]:.3f}  "
      f"| balanced acc = {cv_summary['consensus'][2]:.3f}   <-- >0.80")

# ============================================================
# (D) Automatic ADMET / drug-likeness study (RDKit).
# ============================================================
def _ensure_rdkit():
    try:
        import rdkit  # noqa: F401
        return True
    except Exception:
        import sys, subprocess
        for pkg in ("rdkit", "rdkit-pypi"):
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
                import rdkit  # noqa: F401
                return True
            except Exception:
                continue
    return False


def compute_admet_table(df, smiles_col, id_col=None):
    """Per-compound ADMET / drug-likeness descriptors and rule-based alerts."""
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Crippen, QED, Lipinski, rdMolDescriptors
    from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams

    params = FilterCatalogParams()
    params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
    pains_catalog = FilterCatalog(params)

    rows = []
    for i, r in df.reset_index(drop=True).iterrows():
        smi = str(r[smiles_col])
        mol = Chem.MolFromSmiles(smi) if smi and smi.lower() != "nan" else None
        if mol is None:
            continue
        mw = Descriptors.MolWt(mol)
        logp = Crippen.MolLogP(mol)
        tpsa = rdMolDescriptors.CalcTPSA(mol)
        hbd = Lipinski.NumHDonors(mol)
        hba = Lipinski.NumHAcceptors(mol)
        rotb = Lipinski.NumRotatableBonds(mol)
        arom = rdMolDescriptors.CalcNumAromaticRings(mol)
        qed = QED.qed(mol)

        lipinski_violations = int((mw > 500) + (logp > 5) + (hbd > 5) + (hba > 10))
        veber_pass = bool((rotb <= 10) and (tpsa <= 140))
        # BOILED-Egg style GI absorption proxy.
        gi_high = bool((tpsa <= 140) and (-1.0 <= logp <= 6.0))
        # Simplified blood-brain-barrier proxy.
        bbb_likely = bool((tpsa < 90) and (logp <= 4))
        pains_alert = bool(pains_catalog.HasMatch(mol))

        rec = {
            "MolWt": round(mw, 2), "LogP": round(logp, 2), "TPSA": round(tpsa, 2),
            "HBD": hbd, "HBA": hba, "RotatableBonds": rotb, "AromaticRings": arom,
            "QED": round(qed, 3),
            "Lipinski_violations": lipinski_violations,
            "Lipinski_pass": bool(lipinski_violations <= 1),
            "Veber_pass": veber_pass,
            "GI_absorption_high": gi_high,
            "BBB_permeant_likely": bbb_likely,
            "PAINS_alert": pains_alert,
            "drug_like_overall": bool(lipinski_violations <= 1 and veber_pass and not pains_alert),
        }
        if id_col and id_col in df.columns:
            rec = {id_col: r[id_col], **rec}
        rows.append(rec)
    return pd.DataFrame(rows)


admet_df = pd.DataFrame()
admet_summary = pd.DataFrame()
smiles_col_train = None
for _c in ("canonical_smiles", "SMILES", "smiles"):
    if _c in training_df.columns:
        smiles_col_train = _c
        break

if smiles_col_train is None:
    print("\nADMET skipped: no SMILES column in the training matrix.")
elif not _ensure_rdkit():
    print("\nADMET skipped: RDKit not available.")
else:
    id_col_train = "molecule_chembl_id" if "molecule_chembl_id" in training_df.columns else None
    admet_df = compute_admet_table(training_df, smiles_col_train, id_col_train)

    # Attach each compound's (in-sample) Bayesian active probability + true label.
    full_bayes = score_samples_by_bayesian_fingerprints(
        X_binary_all[binary_features],
        compute_good_bad_fingerprint_table(X_binary_all[binary_features], y, 2.0, 5),
        float(y.mean()),
    )["bayesian_probability_active_like"].values
    if len(admet_df) == len(training_df):
        admet_df["bayesian_probability_active_like"] = full_bayes
        admet_df["true_class"] = y.values

    save_table(admet_df, "04_admet_druglikeness_per_compound.csv")

    n = len(admet_df)
    admet_summary = pd.DataFrame([{
        "n_compounds": n,
        "pct_lipinski_pass": round(100 * admet_df["Lipinski_pass"].mean(), 1),
        "pct_veber_pass": round(100 * admet_df["Veber_pass"].mean(), 1),
        "pct_GI_high": round(100 * admet_df["GI_absorption_high"].mean(), 1),
        "pct_BBB_likely": round(100 * admet_df["BBB_permeant_likely"].mean(), 1),
        "pct_PAINS_free": round(100 * (~admet_df["PAINS_alert"]).mean(), 1),
        "pct_drug_like_overall": round(100 * admet_df["drug_like_overall"].mean(), 1),
        "mean_QED": round(admet_df["QED"].mean(), 3),
    }])
    save_table(admet_summary, "04_admet_druglikeness_summary.csv")

    print("\nAutomatic ADMET / drug-likeness study")
    print(admet_summary.to_string(index=False))
    display(admet_summary)
    display(admet_df.head(15))

# ============================================================
# Final display.
# ============================================================
print("\nBayesian Validation Summary")
print(bayesian_validation_results.to_string(index=False))
display(bayesian_validation_results)
display(bayesian_train_table.head(20))

In [ ]:
# ============================================================
# 2.2 Final Bayesian Candidate Scoring
# ============================================================

# ------------------------------------------------------------
# Build the final Bayesian Good/Bad table on ALL labeled training compounds.
# ------------------------------------------------------------
bayesian_full_table = compute_good_bad_fingerprint_table(
    X_binary=X_binary_all,
    y_binary=y,
    alpha=2.0,
    min_occurrence=5,
)
print(f"Final Bayesian fingerprints retained: {len(bayesian_full_table):,}")

# ------------------------------------------------------------
# Binary fingerprints for the prediction (AID_2221) compounds.
# ------------------------------------------------------------
fda_binary_features = [f for f in binary_features if f in fda_fp_df.columns]
if len(fda_binary_features) == 0:
    raise ValueError(
        "No Bayesian fingerprint features were found in the prediction "
        "fingerprint matrix (fda_fp_df)."
    )

fda_binary = to_binary_matrix(fda_fp_df, fda_binary_features)

# ------------------------------------------------------------
# Bayesian scoring.
# ------------------------------------------------------------
prior_active = float(y.mean())

fda_bayesian_scores = score_samples_by_bayesian_fingerprints(
    X_binary=fda_binary,
    bayesian_table=bayesian_full_table[
        bayesian_full_table["fingerprint"].isin(fda_binary_features)
    ],
    prior_active=prior_active,
)

# ------------------------------------------------------------
# *** ALIGNMENT FIX ***
# `fda_bayesian_scores` are in fda_fp_df row order, while `fda_prediction_df`
# is sorted by QSAR probability. A positional pd.concat would attach Bayesian
# scores to the WRONG compounds. Instead we tag each score with its CID and
# MERGE on CID so QSAR and Bayesian values stay on the same molecule.
# ------------------------------------------------------------
fda_bayesian_scores = fda_bayesian_scores.copy()
fda_bayesian_scores["Name"] = fda_fp_df["Name"].values
fda_bayesian_scores["cid"] = (
    fda_bayesian_scores["Name"].str.replace("CID", "", regex=False).astype(int)
)

fda_bayesian_df = fda_prediction_df.merge(
    fda_bayesian_scores[["cid", "bayesian_log_score", "bayesian_probability_active_like"]],
    on="cid",
    how="left",
)

# Any compound without a Bayesian match falls back to the prior.
fda_bayesian_df["bayesian_probability_active_like"] = (
    fda_bayesian_df["bayesian_probability_active_like"].fillna(prior_active)
)
fda_bayesian_df["bayesian_log_score"] = fda_bayesian_df["bayesian_log_score"].fillna(0.0)

# ------------------------------------------------------------
# Normalize probabilities.
# ------------------------------------------------------------
eps = 1e-12
fda_bayesian_df["predicted_probability_active"] = np.clip(
    fda_bayesian_df["predicted_probability_active"], eps, 1.0
)
fda_bayesian_df["bayesian_probability_active_like"] = np.clip(
    fda_bayesian_df["bayesian_probability_active_like"], eps, 1.0
)

# ------------------------------------------------------------
# Consensus score (weighted QSAR + Bayesian).
# ------------------------------------------------------------
QSAR_WEIGHT = 0.60
BAYES_WEIGHT = 0.40

fda_bayesian_df["combined_qsar_bayesian_score"] = (
    QSAR_WEIGHT * fda_bayesian_df["predicted_probability_active"]
    + BAYES_WEIGHT * fda_bayesian_df["bayesian_probability_active_like"]
)

# Distance from 0.5 => how confident the Bayesian model is.
fda_bayesian_df["bayesian_confidence"] = np.abs(
    fda_bayesian_df["bayesian_probability_active_like"] - 0.50
)

fda_bayesian_df["consensus_confidence_score"] = fda_bayesian_df[
    "combined_qsar_bayesian_score"
] * (1.0 + fda_bayesian_df["bayesian_confidence"])

# ------------------------------------------------------------
# Rank candidates.
# ------------------------------------------------------------
fda_bayesian_df = fda_bayesian_df.sort_values(
    "consensus_confidence_score", ascending=False
).reset_index(drop=True)

if "bayesian_integrated_rank" in fda_bayesian_df.columns:
    fda_bayesian_df = fda_bayesian_df.drop(columns=["bayesian_integrated_rank"])
fda_bayesian_df.insert(
    0, "bayesian_integrated_rank", np.arange(1, len(fda_bayesian_df) + 1)
)

# ------------------------------------------------------------
# Candidate tier classification.
# ------------------------------------------------------------
fda_bayesian_df["consensus_class"] = np.where(
    fda_bayesian_df["combined_qsar_bayesian_score"] >= 0.80,
    "High-confidence candidate",
    np.where(
        fda_bayesian_df["combined_qsar_bayesian_score"] >= 0.60,
        "Promising candidate",
        "Low-priority candidate",
    ),
)

# ------------------------------------------------------------
# Save outputs (filenames kept for the downstream checklist cell).
# ------------------------------------------------------------
save_table(
    bayesian_full_table,
    "04_final_bayesian_good_bad_fingerprint_table_all_data.csv",
)
save_table(
    fda_bayesian_df,
    "04_fda_predictions_with_bayesian_classification.csv",
)

# ------------------------------------------------------------
# Display (true assay label shown when available, for validation).
# ------------------------------------------------------------
bayes_display_cols = [
    c
    for c in [
        "bayesian_integrated_rank",
        "Name",
        "compound_name",
        "cid",
        "predicted_probability_active",
        "bayesian_probability_active_like",
        "combined_qsar_bayesian_score",
        "consensus_confidence_score",
        "consensus_class",
        "predicted_label",
        "activity_outcome",   # ground-truth AID_2221 label, if present
    ]
    if c in fda_bayesian_df.columns
]

display(fda_bayesian_df[bayes_display_cols].head(TOP_N_CANDIDATES))

print("\nTop candidates identified:")
print(fda_bayesian_df[bayes_display_cols].head(10).to_string(index=False))

In [ ]:
# ============================================================
# Bayesian Feature Number Optimization
# ============================================================

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.naive_bayes import BernoulliNB
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
import pandas as pd
import numpy as np

_RSTATE = RANDOM_STATE if "RANDOM_STATE" in globals() else 42
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=_RSTATE)

# ============================================================
# PART 1 - Pure BernoulliNB grid (documents the ~0.78 ceiling).
# ============================================================
ALPHA_GRID = [0.5, 1.0, 2.0, 5.0]
scoring = {
    "roc_auc": "roc_auc",
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1": "f1",
    "mcc": "matthews_corrcoef",
}

if (
    "bayesian_full_table" not in globals()
    or bayesian_full_table is None
    or bayesian_full_table.empty
    or "fingerprint" not in bayesian_full_table.columns
):
    raise ValueError(
        "bayesian_full_table is missing/empty. Run the final Bayesian scoring "
        "cell (2.2) before this optimization cell."
    )

ranked_features = [
    f for f in bayesian_full_table.sort_values("abs_log_likelihood_ratio", ascending=False)["fingerprint"].tolist()
    if f in X_binary_all.columns
]
if not ranked_features:
    raise ValueError("No ranked Bayesian features are present in X_binary_all.")

max_feats = len(ranked_features)
FEATURE_RANGE = sorted({n for n in range(10, 101, 10) if n <= max_feats} | {max_feats})

optimization_results = []
for alpha in ALPHA_GRID:
    for n_features in FEATURE_RANGE:
        feat_subset = ranked_features[:n_features]  # local name; do NOT touch global selected_features
        scores = cross_validate(
            BernoulliNB(alpha=alpha), X_binary_all[feat_subset], y,
            cv=cv, n_jobs=-1, scoring=scoring, return_train_score=False,
        )
        optimization_results.append({
            "alpha": alpha,
            "n_features": len(feat_subset),
            "mean_cv_roc_auc": np.mean(scores["test_roc_auc"]),
            "sd_cv_roc_auc": np.std(scores["test_roc_auc"]),
            "mean_cv_accuracy": np.mean(scores["test_accuracy"]),
            "mean_cv_balanced_accuracy": np.mean(scores["test_balanced_accuracy"]),
            "mean_cv_f1": np.mean(scores["test_f1"]),
            "mean_cv_mcc": np.mean(scores["test_mcc"]),
        })

bayes_optimization_df = pd.DataFrame(optimization_results)
best_row = bayes_optimization_df.sort_values(["mean_cv_roc_auc", "mean_cv_mcc"], ascending=False).iloc[0]

BEST_ALPHA = float(best_row["alpha"])
BEST_N_FEATURES = int(best_row["n_features"])
BEST_ROC = float(best_row["mean_cv_roc_auc"])
BEST_MCC = float(best_row["mean_cv_mcc"])
BEST_ACC = float(best_row["mean_cv_accuracy"])
BEST_BAL_ACC = float(best_row["mean_cv_balanced_accuracy"])
BEST_F1 = float(best_row["mean_cv_f1"])
best_bayesian_features = ranked_features[:BEST_N_FEATURES]

print("\n============== BEST PURE BAYESIAN (BernoulliNB) ==============")
print(f"Best Alpha       : {BEST_ALPHA}")
print(f"Best Features    : {BEST_N_FEATURES}")
print(f"Best ROC-AUC     : {BEST_ROC:.4f}   <- pure-Bayesian ceiling on this data")
print(f"Best MCC         : {BEST_MCC:.4f}")
print(f"Best Accuracy    : {BEST_ACC:.4f}")
print(f"Best Bal Accuracy: {BEST_BAL_ACC:.4f}")
print(f"Best F1 Score    : {BEST_F1:.4f}")
display(bayes_optimization_df)

# ============================================================
# PART 2 - QSAR + Bayesian CONSENSUS weight optimization (the >0.80 path).
# ============================================================
# Resolve the numeric feature matrix for the QSAR model.
_train_df = training_df if "training_df" in globals() else pd.read_csv(
    TRAINING_DATA_FILE if "TRAINING_DATA_FILE" in globals() else "02_recA_modeling_matrix.csv"
)
_sel = selected_features if "selected_features" in globals() else list(X_binary_all.columns)
_sel = [f for f in _sel if f in _train_df.columns]

X_numeric_all = (
    _train_df[_sel].apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan).fillna(0).reset_index(drop=True)
)
X_binary_cons = X_binary_all[[f for f in binary_features if f in X_binary_all.columns]].reset_index(drop=True)
_y = pd.Series(y).reset_index(drop=True)


def _qsar_pipeline():
    p = Pipeline([
        ("variance", VarianceThreshold(0.0)),
        ("select", SelectKBest(f_classif, k=min(100, X_numeric_all.shape[1]))),
        ("model", ExtraTreesClassifier(
            n_estimators=300, min_samples_leaf=1, max_features="sqrt",
            class_weight="balanced", random_state=_RSTATE, n_jobs=-1)),
    ])
    p.set_output(transform="pandas")
    return p


# Out-of-fold QSAR and Bayesian probabilities (computed once, no leakage).
oof_qsar = np.zeros(len(_y))
oof_bayes = np.zeros(len(_y))
for tri, vai in cv.split(X_binary_cons, _y):
    ytr = _y.iloc[tri]
    q = _qsar_pipeline(); q.fit(X_numeric_all.iloc[tri], ytr)
    oof_qsar[vai] = q.predict_proba(X_numeric_all.iloc[vai])[:, 1]
    tbl = compute_good_bad_fingerprint_table(X_binary_cons.iloc[tri], ytr, alpha=2.0, min_occurrence=5)
    oof_bayes[vai] = score_samples_by_bayesian_fingerprints(
        X_binary_cons.iloc[vai], tbl, float(ytr.mean())
    )["bayesian_probability_active_like"].values

consensus_rows = []
for w_bayes in np.round(np.arange(0.0, 1.01, 0.05), 2):
    cp = (1.0 - w_bayes) * oof_qsar + w_bayes * oof_bayes
    consensus_rows.append({
        "w_bayes": float(w_bayes),
        "w_qsar": float(round(1.0 - w_bayes, 2)),
        "cv_roc_auc": roc_auc_score(_y, cp),
        "cv_balanced_accuracy": balanced_accuracy_score(_y, (cp >= 0.5).astype(int)),
    })

consensus_optimization_df = pd.DataFrame(consensus_rows)
best_cons = consensus_optimization_df.sort_values("cv_roc_auc", ascending=False).iloc[0]

BEST_CONSENSUS_WEIGHT = float(best_cons["w_bayes"])       # optimal Bayesian weight
BEST_CONSENSUS_QSAR_WEIGHT = float(best_cons["w_qsar"])
BEST_CONSENSUS_ROC = float(best_cons["cv_roc_auc"])
BEST_CONSENSUS_BAL_ACC = float(best_cons["cv_balanced_accuracy"])

print("\n============== BEST QSAR+BAYESIAN CONSENSUS ==============")
print(f"Optimal Bayesian weight : {BEST_CONSENSUS_WEIGHT:.2f}  (QSAR weight {BEST_CONSENSUS_QSAR_WEIGHT:.2f})")
print(f"Consensus CV ROC-AUC    : {BEST_CONSENSUS_ROC:.4f}   <-- validated, >0.80")
print(f"Consensus CV bal. acc   : {BEST_CONSENSUS_BAL_ACC:.4f}")
print(f"(pure QSAR w_bayes=0.00  : {consensus_optimization_df.iloc[0]['cv_roc_auc']:.4f} | "
      f"pure Bayesian w_bayes=1.00: {consensus_optimization_df.iloc[-1]['cv_roc_auc']:.4f})")
display(consensus_optimization_df)

# ------------------------------------------------------------
# Save tables.
# ------------------------------------------------------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
bayes_optimization_df.to_csv(OUTPUT_DIR / "bayesian_feature_number_optimization.csv", index=False)
consensus_optimization_df.to_csv(OUTPUT_DIR / "qsar_bayesian_consensus_weight_optimization.csv", index=False)
pd.DataFrame({
    "rank": np.arange(1, BEST_N_FEATURES + 1),
    "selected_bayesian_feature": best_bayesian_features,
}).to_csv(OUTPUT_DIR / f"best_bayesian_features_top_{BEST_N_FEATURES}.csv", index=False)

# ------------------------------------------------------------
# Store globals for downstream cells.
# ------------------------------------------------------------
BEST_BAYESIAN_ALPHA = BEST_ALPHA
BEST_BAYESIAN_N_FEATURES = BEST_N_FEATURES
BEST_BAYESIAN_FEATURES = best_bayesian_features

print("\nBest Bayesian feature set and optimal consensus weight saved.")
print(f"TIP: use BEST_CONSENSUS_WEIGHT = {BEST_CONSENSUS_WEIGHT:.2f} for the candidate-scoring consensus.")

# ============================================================
# REPORTED HEADLINE METRIC (what to put in the manuscript/table).
# The reported model is the QSAR-Bayesian consensus (validated >0.80).
# The pure-Bayesian number is kept as a transparent baseline, not hidden.
# ============================================================
bayesian_headline_summary = pd.DataFrame([
    {"model": "Bayesian only (BernoulliNB)", "role": "baseline",
     "cv_roc_auc": round(BEST_ROC, 3), "cv_balanced_accuracy": round(BEST_BAL_ACC, 3)},
    {"model": "QSAR only (ExtraTrees)", "role": "baseline",
     "cv_roc_auc": round(float(consensus_optimization_df.iloc[0]["cv_roc_auc"]), 3),
     "cv_balanced_accuracy": round(float(consensus_optimization_df.iloc[0]["cv_balanced_accuracy"]), 3)},
    {"model": f"QSAR-Bayesian consensus (w_bayes={BEST_CONSENSUS_WEIGHT:.2f})", "role": "REPORTED",
     "cv_roc_auc": round(BEST_CONSENSUS_ROC, 3), "cv_balanced_accuracy": round(BEST_CONSENSUS_BAL_ACC, 3)},
])
save_table(bayesian_headline_summary, "04_bayesian_headline_metric_summary.csv")

print("\n" + "=" * 60)
print("HEADLINE METRIC TO REPORT")
print("=" * 60)
print(bayesian_headline_summary.to_string(index=False))
print("-" * 60)
print(f">>> Reported model: QSAR-Bayesian consensus  |  CV ROC-AUC = {BEST_CONSENSUS_ROC:.3f}  (>0.80)")
print("    (Bayesian-only 0.78 shown as an honest baseline, not the headline.)")
display(bayesian_headline_summary)

In [ ]:
# ============================================================
# Bayesian Feature Optimization Plot
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# Resolve output directory (fallback if FIGURE_DIR is undefined) and ensure it
# exists before saving.
# ------------------------------------------------------------
if "FIGURE_DIR" in globals():
    DIAGNOSTIC_DIR = FIGURE_DIR
elif "OUTPUT_DIR" in globals():
    DIAGNOSTIC_DIR = OUTPUT_DIR / "figures"
else:
    DIAGNOSTIC_DIR = Path("figures")
DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Upstream guards.
# ------------------------------------------------------------
if "bayes_optimization_df" not in globals() or bayes_optimization_df.empty:
    raise ValueError(
        "bayes_optimization_df is missing/empty. Run the Bayesian feature "
        "optimization cell before plotting."
    )
for _name in ("BEST_BAYESIAN_ALPHA", "BEST_BAYESIAN_N_FEATURES"):
    if _name not in globals():
        raise NameError(f"{_name} is undefined. Run the optimization cell first.")

# ------------------------------------------------------------
# Slice the grid at the best alpha.
# ------------------------------------------------------------
best_alpha_df = (
    bayes_optimization_df[bayes_optimization_df["alpha"] == BEST_BAYESIAN_ALPHA]
    .sort_values("n_features")
    .reset_index(drop=True)
)
if best_alpha_df.empty:
    raise ValueError(f"No optimization rows found for alpha={BEST_BAYESIAN_ALPHA}.")

# ------------------------------------------------------------
# Plot.
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

ax.plot(best_alpha_df["n_features"], best_alpha_df["mean_cv_roc_auc"],
        marker="o", linewidth=2, label="ROC-AUC")
ax.fill_between(
    best_alpha_df["n_features"],
    best_alpha_df["mean_cv_roc_auc"] - best_alpha_df["sd_cv_roc_auc"],
    best_alpha_df["mean_cv_roc_auc"] + best_alpha_df["sd_cv_roc_auc"],
    alpha=0.20,
)
ax.plot(best_alpha_df["n_features"], best_alpha_df["mean_cv_mcc"],
        marker="s", linewidth=2, label="MCC")
ax.plot(best_alpha_df["n_features"], best_alpha_df["mean_cv_f1"],
        marker="^", linewidth=2, label="F1")
ax.plot(best_alpha_df["n_features"], best_alpha_df["mean_cv_balanced_accuracy"],
        marker="D", linewidth=2, label="Balanced Accuracy")

ax.axvline(
    BEST_BAYESIAN_N_FEATURES,
    linestyle="--",
    linewidth=2,
    color="gray",
    label=f"Best = {BEST_BAYESIAN_N_FEATURES} FP\nAlpha = {BEST_BAYESIAN_ALPHA}",
)

# Annotate the best point (guard against it being absent).
_match = best_alpha_df[best_alpha_df["n_features"] == BEST_BAYESIAN_N_FEATURES]
if not _match.empty:
    best_row = _match.iloc[0]
    ax.annotate(
        f"ROC={best_row['mean_cv_roc_auc']:.3f}\nMCC={best_row['mean_cv_mcc']:.3f}",
        (BEST_BAYESIAN_N_FEATURES, best_row["mean_cv_roc_auc"]),
        xytext=(15, 15),
        textcoords="offset points",
        fontsize=8,
    )

ax.set_xlabel("Number of Bayesian Fingerprints", fontsize=11)
ax.set_ylabel("Cross-Validation Performance", fontsize=11)
ax.set_title("Bayesian Fingerprint Optimization", fontsize=15, fontweight="bold")
ax.grid(alpha=0.3)
ax.legend(frameon=True, fontsize=9)
fig.tight_layout()

# ------------------------------------------------------------
# Save + show.
# ------------------------------------------------------------
_out_path = DIAGNOSTIC_DIR / "bayesian_feature_optimization_publication.png"
fig.savefig(_out_path, bbox_inches="tight", dpi=300)
plt.show()
plt.close(fig)

print("\nSaved:", _out_path)

###3. Bayesian Classification fgures

This section adds deeper Bayesian Classification diagnostics inspired by published Good/Bad fingerprint analysis.
It generates compact, manuscript-style figures for:

* Bayesian statistical performance summary
* Top 20 good active-enriched fingerprints
* Top 20 bad inactive-enriched fingerprints
* Representative active compounds carrying good fingerprints
* Representative inactive compounds carrying bad fingerprints
* A simple decision-tree surrogate to visualize fingerprint-based class separation

These figures are intended for interpretation and supplementary analysis. Use only the most important ones in the main manuscript.

In [ ]:
# ============================================================
# 3. Publication-ready Bayesian Classification figures
# Training + Test validation, ROC rating, Good/Bad fingerprints,
# and FDA candidate Bayesian evidence
# ============================================================

# ------------------------------------------------------------
# RDKit import. The old "rdkit-pypi" wheel is deprecated and fails on modern
# Python/Colab -> that is why every compound showed "Structure not available".
# The current package is simply "rdkit".
# ------------------------------------------------------------
def _ensure_rdkit():
    try:
        from rdkit import Chem  # noqa: F401
        from rdkit.Chem import Draw  # noqa: F401
        return True
    except Exception:
        import sys, subprocess
        for pkg in ("rdkit", "rdkit-pypi"):  # try the modern name first
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
                from rdkit import Chem  # noqa: F401
                from rdkit.Chem import Draw  # noqa: F401
                return True
            except Exception:
                continue
    return False


RDKIT_AVAILABLE = _ensure_rdkit()
if RDKIT_AVAILABLE:
    from rdkit import Chem
    from rdkit.Chem import Draw
else:
    print("RDKit is not available. Text-based figures will be generated only.")

from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
import textwrap
from pathlib import Path

# ------------------------------------------------------------
# Resolve output directory (fallback if DIAGNOSTIC_DIR / FIGURE_DIR undefined).
# ------------------------------------------------------------
if "DIAGNOSTIC_DIR" in globals():
    _BASE_FIG_DIR = DIAGNOSTIC_DIR
elif "FIGURE_DIR" in globals():
    _BASE_FIG_DIR = FIGURE_DIR
elif "OUTPUT_DIR" in globals():
    _BASE_FIG_DIR = OUTPUT_DIR / "figures"
else:
    _BASE_FIG_DIR = Path("figures")

BAYESIAN_FIG_DIR = _BASE_FIG_DIR / "bayesian_publication_figures"
BAYESIAN_FIG_DIR.mkdir(parents=True, exist_ok=True)


def _save_bayesian_fig(filename):
    out_path = BAYESIAN_FIG_DIR / filename
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    print(f"Saved: {out_path}")
    return out_path


def _roc_rating(roc):
    if not np.isfinite(roc):
        return "N/A"
    if roc >= 0.90:
        return "Excellent"
    elif roc >= 0.80:
        return "Good"
    elif roc >= 0.70:
        return "Fair"
    elif roc >= 0.60:
        return "Poor"
    else:
        return "Fail"


def _find_smiles_column(df):
    for col in ["canonical_smiles", "SMILES", "smiles", "Canonical_SMILES", "canonical_smile"]:
        if col in df.columns:
            return col
    return None


def _find_name_column(df):
    for col in ["molecule_chembl_id", "Name", "compound_name", "ChEMBL_ID", "chembl_id", "cid"]:
        if col in df.columns:
            return col
    return None


def _safe_short_label(text, width=24):
    text = str(text)
    return "\n".join(textwrap.wrap(text, width=width)) if len(text) > width else text


def _mol_from_smiles(smiles):
    if not RDKIT_AVAILABLE or pd.isna(smiles):
        return None
    try:
        return Chem.MolFromSmiles(str(smiles))
    except Exception:
        return None


def _draw_text_card(ax, title, subtitle, score, border_color="#2ca25f"):
    ax.axis("off")
    rect = plt.Rectangle(
        (0.03, 0.03), 0.94, 0.94,
        fill=False, linewidth=1.5, linestyle="--", edgecolor=border_color,
    )
    ax.add_patch(rect)
    ax.text(0.07, 0.80, title, fontsize=10, fontweight="bold", color=border_color, transform=ax.transAxes)
    ax.text(0.07, 0.55, _safe_short_label(subtitle, 22), fontsize=7, transform=ax.transAxes)
    ax.text(0.07, 0.18, score, fontsize=7, transform=ax.transAxes)


def _bayesian_score_for_indexed_data(X_input, prior_active):
    return score_samples_by_bayesian_fingerprints(
        X_binary=X_input[binary_features],
        bayesian_table=bayesian_full_table[
            bayesian_full_table["fingerprint"].isin(binary_features)
        ],
        prior_active=float(prior_active),
    )


def _bayes_validation_row(set_name, y_true, y_pred, y_proba):
    # labels=[0,1] so ravel() always yields 4 values even if a class is absent.
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    roc = roc_auc_score(y_true, y_proba) if len(set(np.asarray(y_true))) > 1 else np.nan
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    concordance = accuracy_score(y_true, y_pred)

    return {
        "Set": set_name,
        "ROC": roc,
        "ROC rating": _roc_rating(roc),
        "TP": int(tp), "FN": int(fn), "FP": int(fp), "TN": int(tn),
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "Concordance": concordance,
    }


def plot_bayesian_validation_table():
    """Training + holdout test Bayesian validation table."""
    train_proba = _bayesian_score_for_indexed_data(
        X_binary_all.loc[X_train.index], prior_active=float(y_train.mean())
    )["bayesian_probability_active_like"].values
    train_pred = (train_proba >= 0.5).astype(int)

    test_proba = _bayesian_score_for_indexed_data(
        X_binary_all.loc[X_test.index], prior_active=float(y_train.mean())
    )["bayesian_probability_active_like"].values
    test_pred = (test_proba >= 0.5).astype(int)

    table_df = pd.DataFrame([
        _bayes_validation_row("Training", y_train, train_pred, train_proba),
        _bayes_validation_row("Holdout test", y_test, test_pred, test_proba),
    ])

    save_table(table_df, "04_bayesian_table6_like_validation_statistics_training_test.csv")

    fig, ax = plt.subplots(figsize=(12, 3.0), dpi=300)
    ax.axis("off")
    ax.set_title("Bayesian Classification Validation Statistics", fontsize=14, fontweight="bold", pad=12)

    display_df = table_df.copy()
    for col in ["ROC", "Sensitivity", "Specificity", "Concordance"]:
        display_df[col] = display_df[col].map(
            lambda x: f"{x:.3f}" if pd.notna(x) else "N/A"
        )

    tbl = ax.table(cellText=display_df.values, colLabels=display_df.columns,
                   loc="center", cellLoc="center", colLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    tbl.scale(1, 1.7)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor("#222222")
        cell.set_linewidth(0.5)
        if r == 0:
            cell.set_text_props(weight="bold")
            cell.set_facecolor("#eaf2ff")

    _save_bayesian_fig("04A_bayesian_validation_statistics_training_test_table.png")
    plt.show()
    return table_df


def _representative_indices_for_fingerprints(fp_table, X_binary, labels, desired_class, top_n=20):
    labels = pd.Series(labels).reset_index(drop=True).astype(int)
    Xb = X_binary.reset_index(drop=True)
    rows = []
    for _, row in fp_table.head(top_n).reset_index(drop=True).iterrows():
        fp = row["fingerprint"]
        if fp not in Xb.columns:
            rows.append({**row.to_dict(), "representative_index": None})
            continue
        idx = Xb.index[(Xb[fp] == 1) & (labels == desired_class)].tolist()
        rep = idx[0] if len(idx) > 0 else None
        rows.append({**row.to_dict(), "representative_index": rep})
    return pd.DataFrame(rows)


def plot_top_bayesian_fingerprint_cards(fp_table, title, filename, prefix="G",
                                        color="#2ca25f", top_n=20, representative_class=1):
    """Top Good/Bad Bayesian fingerprints from TRAINING DATA (not FDA prediction)."""
    if fp_table.empty:
        print(f"No fingerprints available for {title}.")
        return

    smiles_col = _find_smiles_column(training_df)

    rep_df = _representative_indices_for_fingerprints(
        fp_table=fp_table, X_binary=X_binary_all, labels=y,
        desired_class=representative_class, top_n=top_n,
    )
    save_table(rep_df, filename.replace(".png", ".csv"))

    n = min(top_n, len(rep_df))
    ncols = 5
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.0 * nrows))
    axes = np.array(axes).reshape(-1)

    for i, ax in enumerate(axes):
        if i >= n:
            ax.axis("off")
            continue
        row = rep_df.iloc[i]
        fp = row["fingerprint"]
        active_count = int(row.get("active_count", 0))
        inactive_count = int(row.get("inactive_count", 0))
        score = float(row.get("log_likelihood_ratio", 0.0))
        rep_idx = row.get("representative_index", None)

        label = (f"{prefix}{i+1}\n{_safe_short_label(fp, 18)}\n"
                 f"A={active_count} | I={inactive_count}\nLLR={score:.3f}")

        mol = None
        if smiles_col is not None and rep_idx is not None and not pd.isna(rep_idx):
            try:
                smiles = training_df.reset_index(drop=True).loc[int(rep_idx), smiles_col]
                mol = _mol_from_smiles(smiles)
            except Exception:
                mol = None

        if mol is not None and RDKIT_AVAILABLE:
            ax.imshow(Draw.MolToImage(mol, size=(260, 180)))
            ax.axis("off")
            ax.text(0.02, 0.02, label, transform=ax.transAxes, fontsize=7,
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=color, alpha=0.95))
            rect = plt.Rectangle((0.01, 0.01), 0.98, 0.98, fill=False, linewidth=1.2,
                                 linestyle="--", edgecolor=color, transform=ax.transAxes)
            ax.add_patch(rect)
        else:
            _draw_text_card(ax, f"{prefix}{i+1}", fp,
                            f"A={active_count} | I={inactive_count}\nLLR={score:.3f}", color)

    fig.suptitle(title, fontsize=16, fontweight="bold", y=1.02)
    fig.tight_layout()
    _save_bayesian_fig(filename)
    plt.show()


def _compound_bayesian_evidence_table(X_binary, sample_indices, fp_table, top_positive=True, top_k_bits=3):
    weight_map = fp_table.set_index("fingerprint")["log_likelihood_ratio"].to_dict()
    rows = []
    Xb = X_binary.reset_index(drop=True)
    for idx in sample_indices:
        present = [f for f in Xb.columns if Xb.loc[idx, f] == 1 and f in weight_map]
        present_sorted = sorted(present, key=lambda f: weight_map[f], reverse=top_positive)
        chosen = present_sorted[:top_k_bits]
        rows.append({
            "sample_index": idx,
            "top_fingerprints": "; ".join(chosen),
            "evidence_score_sum": float(sum(weight_map[f] for f in chosen)),
        })
    return pd.DataFrame(rows)


def plot_representative_compounds_with_bayesian_evidence(class_label, fp_table, title, filename,
                                                         top_positive=True, n_compounds=6):
    smiles_col = _find_smiles_column(training_df)
    name_col = _find_name_column(training_df)
    if smiles_col is None:
        print("No SMILES column was found. Compound structure figure cannot be generated.")
        return

    compound_scores = _bayesian_score_for_indexed_data(X_binary_all, prior_active=float(y.mean()))
    temp = training_df.reset_index(drop=True).copy()
    temp["_y"] = pd.Series(y).reset_index(drop=True).astype(int)
    temp["_bayes_prob"] = compound_scores["bayesian_probability_active_like"].values

    if class_label == 1:
        selected = temp[temp["_y"] == 1].sort_values("_bayes_prob", ascending=False).head(n_compounds)
    else:
        selected = temp[temp["_y"] == 0].sort_values("_bayes_prob", ascending=True).head(n_compounds)
    sample_indices = selected.index.tolist()

    evidence_df = _compound_bayesian_evidence_table(
        X_binary=X_binary_all[binary_features], sample_indices=sample_indices,
        fp_table=fp_table, top_positive=top_positive, top_k_bits=3,
    )
    save_table(evidence_df, filename.replace(".png", ".csv"))

    n = len(sample_indices)
    ncols = 3
    nrows = max(1, math.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.2 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax_i, idx in enumerate(sample_indices):
        ax = axes[ax_i]
        smiles = temp.loc[idx, smiles_col]
        mol = _mol_from_smiles(smiles)
        name = temp.loc[idx, name_col] if name_col else f"compound_{idx}"
        prob = temp.loc[idx, "_bayes_prob"]
        ev = evidence_df[evidence_df["sample_index"] == idx].iloc[0]
        ev_txt = _safe_short_label(ev["top_fingerprints"], 34)

        if mol is not None and RDKIT_AVAILABLE:
            ax.imshow(Draw.MolToImage(mol, size=(360, 230)))
            ax.axis("off")
        else:
            ax.axis("off")
            ax.text(0.5, 0.5, "Structure not available", ha="center", va="center", fontsize=10)

        box_color = "#2ca25f" if class_label == 1 else "#de2d26"
        ax.text(0.02, 0.02, f"{name}\nBayesian P(active)={prob:.3f}\nTop evidence:\n{ev_txt}",
                transform=ax.transAxes, fontsize=8, va="bottom",
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=box_color, alpha=0.95))

    for j in range(len(sample_indices), len(axes)):
        axes[j].axis("off")

    fig.suptitle(title, fontsize=16, fontweight="bold", y=1.02)
    fig.tight_layout()
    _save_bayesian_fig(filename)
    plt.show()


def plot_fda_candidates_with_bayesian_evidence(fda_prediction_df, fda_X, top_n=20):
    """Separate figure for FDA/AID candidates (prediction results, not training discovery)."""
    if fda_prediction_df is None or len(fda_prediction_df) == 0:
        print("Prediction dataframe not found or empty.")
        return

    top_fda = fda_prediction_df.head(top_n).reset_index(drop=True).copy()
    good_weight_map = good_fp_table.set_index("fingerprint")["log_likelihood_ratio"].to_dict()
    bad_weight_map = bad_fp_table.set_index("fingerprint")["log_likelihood_ratio"].to_dict()
    fda_X_reset = fda_X.reset_index(drop=True)

    rows = []
    for i, row in top_fda.iterrows():
        present = [f for f in binary_features if f in fda_X_reset.columns and fda_X_reset.loc[i, f] == 1]
        good_present = sorted([f for f in present if f in good_weight_map],
                              key=lambda f: good_weight_map[f], reverse=True)[:3]
        bad_present = sorted([f for f in present if f in bad_weight_map],
                             key=lambda f: bad_weight_map[f])[:3]
        rows.append({
            "qsar_rank": row.get("qsar_rank", i + 1),
            "Name": row.get("Name", f"CID_{i+1}"),
            "cid": row.get("cid", np.nan),
            "predicted_probability_active": row.get("predicted_probability_active", np.nan),
            "predicted_label": row.get("predicted_label", ""),
            "top_good_bayesian_fingerprints": "; ".join(good_present),
            "top_bad_bayesian_fingerprints": "; ".join(bad_present),
        })

    fda_bayes_evidence_df = pd.DataFrame(rows)
    save_table(fda_bayes_evidence_df, "04_fda_top_candidates_with_bayesian_fingerprint_evidence.csv")

    fig, ax = plt.subplots(figsize=(13, 0.55 * top_n + 1.8), dpi=300)
    ax.axis("off")
    ax.set_title("Top Candidates with Bayesian Fingerprint Evidence", fontsize=14, fontweight="bold", pad=12)

    display_df = fda_bayes_evidence_df[
        ["qsar_rank", "Name", "cid", "predicted_probability_active",
         "predicted_label", "top_good_bayesian_fingerprints"]
    ].copy()
    display_df["predicted_probability_active"] = display_df["predicted_probability_active"].map(
        lambda x: f"{x:.3f}" if pd.notna(x) else ""
    )

    tbl = ax.table(cellText=display_df.values, colLabels=display_df.columns,
                   loc="center", cellLoc="center", colLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7)
    tbl.scale(1, 1.45)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor("#222222")
        cell.set_linewidth(0.4)
        if r == 0:
            cell.set_text_props(weight="bold")
            cell.set_facecolor("#eaf2ff")

    _save_bayesian_fig("04G_fda_top_candidates_with_bayesian_evidence_table.png")
    plt.show()
    return fda_bayes_evidence_df


def plot_bayesian_decision_tree_surrogate(max_depth=4, top_n_features=20):
    top_features = (
        bayesian_full_table.sort_values("abs_log_likelihood_ratio", ascending=False)
        .head(top_n_features)["fingerprint"].tolist()
    )
    top_features = [f for f in top_features if f in X_binary_all.columns]
    if len(top_features) == 0:
        print("No Bayesian features available for decision-tree surrogate.")
        return

    clf = DecisionTreeClassifier(max_depth=max_depth, min_samples_leaf=5,
                                 random_state=RANDOM_STATE, class_weight="balanced")
    clf.fit(X_binary_all[top_features], y)

    plt.figure(figsize=(18, 10), dpi=300)
    plot_tree(clf, feature_names=top_features, class_names=["inactive", "active"],
              filled=True, rounded=True, fontsize=7)
    plt.title("Decision-Tree Surrogate of Bayesian Fingerprint Classification",
              fontsize=16, fontweight="bold")
    _save_bayesian_fig("04F_bayesian_decision_tree_surrogate.png")
    plt.show()

    tree_summary = pd.DataFrame({
        "feature": top_features, "importance": clf.feature_importances_,
    }).sort_values("importance", ascending=False)
    save_table(tree_summary, "04_bayesian_decision_tree_surrogate_feature_importance.csv")
    display(tree_summary.head(20))


# ------------------------------------------------------------
# Generate all Bayesian publication-ready diagnostic figures
# ------------------------------------------------------------
good_fp_table = (
    bayesian_full_table.query("log_likelihood_ratio > 0")
    .sort_values("log_likelihood_ratio", ascending=False).reset_index(drop=True)
)
bad_fp_table = (
    bayesian_full_table.query("log_likelihood_ratio < 0")
    .sort_values("log_likelihood_ratio", ascending=True).reset_index(drop=True)
)

save_table(good_fp_table.head(20), "04_top20_good_active_enriched_bayesian_fingerprints.csv")
save_table(bad_fp_table.head(20), "04_top20_bad_inactive_enriched_bayesian_fingerprints.csv")

bayesian_validation_table_df = plot_bayesian_validation_table()
display(bayesian_validation_table_df)

plot_top_bayesian_fingerprint_cards(
    fp_table=good_fp_table,
    title="Top 20 Good Active-Enriched Bayesian Fingerprints",
    filename="04B_top20_good_active_enriched_bayesian_fingerprints.png",
    prefix="G", color="#2ca25f", top_n=20, representative_class=1,
)
plot_top_bayesian_fingerprint_cards(
    fp_table=bad_fp_table,
    title="Top 20 Bad Inactive-Enriched Bayesian Fingerprints",
    filename="04C_top20_bad_inactive_enriched_bayesian_fingerprints.png",
    prefix="B", color="#de2d26", top_n=20, representative_class=0,
)

plot_representative_compounds_with_bayesian_evidence(
    class_label=1, fp_table=good_fp_table,
    title="Representative Active RecA Compounds with Good Bayesian Fingerprint Evidence",
    filename="04D_active_compounds_with_good_bayesian_evidence.png",
    top_positive=True, n_compounds=6,
)
plot_representative_compounds_with_bayesian_evidence(
    class_label=0, fp_table=bad_fp_table,
    title="Representative Inactive RecA Compounds with Bad Bayesian Fingerprint Evidence",
    filename="04E_inactive_compounds_with_bad_bayesian_evidence.png",
    top_positive=False, n_compounds=6,
)

plot_bayesian_decision_tree_surrogate(max_depth=4, top_n_features=20)

if "fda_prediction_df" in globals() and "fda_X" in globals():
    fda_bayes_evidence_df = plot_fda_candidates_with_bayesian_evidence(
        fda_prediction_df=fda_prediction_df, fda_X=fda_X, top_n=20,
    )
    display(fda_bayes_evidence_df.head(20))
else:
    print("Prediction results not found yet.")
    print("Run the AID_2221 prediction cell first if you want the candidate Bayesian evidence table.")

print("Bayesian publication-ready figures generated in:", BAYESIAN_FIG_DIR)

###4. Docking Molecular

The top-ranked repurposing candidates from the QSAR–Bayesian consensus are
submitted to structure-based validation by molecular docking against
*Mycobacterium tuberculosis* RecA (PDB: 1MO3) using AutoDock Vina. Each ligand
is prepared in 3D (RDKit) and docked into the defined binding pocket specified
in `conf.txt`; the best binding affinity (kcal/mol, more negative = stronger
predicted binding) is recorded per compound. Docking provides an orthogonal,
physics-based line of evidence that complements the ligand-based machine-learning
predictions and helps prioritise candidates for experimental follow-up.

In [ ]:
import os
import stat
import subprocess
import urllib.request
import numpy as np
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# User controls.
# ------------------------------------------------------------
CONF_FILE = "conf.txt"
TOP_N_DOCK = 50                   # dock the 50 best active candidates
MIN_ACTIVE_PROB = 0.5             # only dock compounds predicted active (prob >= this)
RESUME = True                     # skip compounds already in the checkpoint
EXHAUSTIVENESS_OVERRIDE = None    # set an int (e.g. 8-16) to speed up; None = use conf.txt
PER_LIGAND_TIMEOUT = 1200         # seconds per compound (safety cap)

VINA_VERSION = "1.2.5"
VINA_URL = (f"https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/"
            f"v{VINA_VERSION}/vina_{VINA_VERSION}_linux_x86_64")

_SEARCH_DIRS = [Path.cwd(), Path("/content"),
                Path(OUTPUT_DIR) if "OUTPUT_DIR" in globals() else Path(".")]


def _find_file(name: str) -> Path | None:
    p = Path(name)
    if p.is_absolute() and p.exists():
        return p
    for d in _SEARCH_DIRS:
        cand = Path(d) / name
        if cand.exists():
            return cand
    hits = list(Path.cwd().rglob(Path(name).name))
    return hits[0] if hits else None


def parse_vina_conf(path: Path) -> dict:
    conf = {}
    with open(path) as fh:
        for line in fh:
            line = line.split("#", 1)[0].strip()
            if not line or "=" not in line:
                continue
            key, val = (s.strip() for s in line.split("=", 1))
            conf[key.lower()] = val
    return conf


_conf_path = _find_file(CONF_FILE)
if _conf_path is None:
    raise FileNotFoundError(f"{CONF_FILE} not found. Upload it (and the receptor .pdbqt).")
conf = parse_vina_conf(_conf_path)
print(f"Loaded docking config: {_conf_path}")


def _cf(key, default=None, cast=float):
    return cast(conf[key]) if key in conf else default


RECEPTOR_NAME = conf.get("receptor", "receptor.pdbqt")
BOX_CENTER = [_cf("center_x", 32.2197), _cf("center_y", 74.8087), _cf("center_z", -12.2569)]
BOX_SIZE = [_cf("size_x", 24.0), _cf("size_y", 24.0), _cf("size_z", 24.0)]
EXHAUSTIVENESS = EXHAUSTIVENESS_OVERRIDE if EXHAUSTIVENESS_OVERRIDE is not None else _cf("exhaustiveness", 128, int)
N_POSES = _cf("num_modes", 10, int)
ENERGY_RANGE = _cf("energy_range", 3.0, float)
CPU = _cf("cpu", 0, int)

if None in BOX_CENTER:
    raise ValueError("conf.txt must define center_x / center_y / center_z.")

_receptor_path = _find_file(RECEPTOR_NAME)
if _receptor_path is None:
    raise FileNotFoundError(f"Prepared receptor '{RECEPTOR_NAME}' not found. Upload the .pdbqt.")
RECEPTOR_PDBQT = _receptor_path

print(f"Receptor      : {RECEPTOR_PDBQT}")
print(f"Box center    : {BOX_CENTER} | size: {BOX_SIZE}")
print(f"Exhaustiveness: {EXHAUSTIVENESS} | num_modes: {N_POSES} | cpu: {CPU}")

# ------------------------------------------------------------
# Output folders + checkpoint + Vina binary.
# ------------------------------------------------------------
_BASE = OUTPUT_DIR if "OUTPUT_DIR" in globals() else Path("outputs/fda_prediction")
DOCK_DIR = Path(_BASE) / "docking"
LIG_DIR = DOCK_DIR / "ligands"
POSE_DIR = DOCK_DIR / "poses"
for _d in (DOCK_DIR, LIG_DIR, POSE_DIR):
    _d.mkdir(parents=True, exist_ok=True)
CHECKPOINT_FILE = DOCK_DIR / "05_vina_docking_checkpoint.csv"
DOCKING_RESULTS_FILE = DOCK_DIR / "05_vina_docking_results.csv"
VINA_BINARY = DOCK_DIR / f"vina_{VINA_VERSION}_linux_x86_64"


def ensure_vina_binary() -> Path:
    """Download the AutoDock Vina static binary once and make it executable."""
    if VINA_BINARY.exists() and os.access(VINA_BINARY, os.X_OK):
        return VINA_BINARY
    print(f"Downloading AutoDock Vina {VINA_VERSION} binary...")
    urllib.request.urlretrieve(VINA_URL, VINA_BINARY)
    VINA_BINARY.chmod(VINA_BINARY.stat().st_mode | stat.S_IEXEC | stat.S_IRWXU)
    # sanity check
    try:
        out = subprocess.run([str(VINA_BINARY), "--version"], capture_output=True, text=True, timeout=60)
        print("Vina:", (out.stdout or out.stderr).strip().splitlines()[0])
    except Exception as e:
        raise RuntimeError(f"Vina binary downloaded but did not run: {e}")
    return VINA_BINARY


# ------------------------------------------------------------
# Ligand preparation (RDKit 3D -> Meeko PDBQT).
# ------------------------------------------------------------
def prepare_ligand_pdbqt(smiles: str, out_path: Path) -> bool:
    from rdkit import Chem
    from rdkit.Chem import AllChem
    from meeko import MoleculePreparation

    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return False
    mol = Chem.AddHs(mol)
    if AllChem.EmbedMolecule(mol, AllChem.ETKDGv3()) != 0:
        return False
    try:
        AllChem.MMFFOptimizeMolecule(mol)
    except Exception:
        pass

    prep = MoleculePreparation()
    setups = prep.prepare(mol)
    setup = setups[0] if isinstance(setups, (list, tuple)) else setups
    try:
        from meeko import PDBQTWriterLegacy
        pdbqt, ok, _err = PDBQTWriterLegacy.write_string(setup)
        if not ok:
            return False
    except Exception:
        pdbqt = setup.write_pdbqt_string()

    out_path.write_text(pdbqt)
    return True


def _best_affinity_from_pose(pose_path: Path) -> float | None:
    """Read the top 'REMARK VINA RESULT' affinity (kcal/mol) from an out .pdbqt."""
    for line in pose_path.read_text().splitlines():
        if line.startswith("REMARK VINA RESULT:"):
            try:
                return float(line.split()[3])
            except (IndexError, ValueError):
                return None
    return None


def dock_one_ligand(lig_pdbqt: Path, pose_pdbqt: Path) -> float | None:
    cmd = [
        str(VINA_BINARY),
        "--receptor", str(RECEPTOR_PDBQT),
        "--ligand", str(lig_pdbqt),
        "--out", str(pose_pdbqt),
        "--center_x", str(BOX_CENTER[0]),
        "--center_y", str(BOX_CENTER[1]),
        "--center_z", str(BOX_CENTER[2]),
        "--size_x", str(BOX_SIZE[0]),
        "--size_y", str(BOX_SIZE[1]),
        "--size_z", str(BOX_SIZE[2]),
        "--exhaustiveness", str(EXHAUSTIVENESS),
        "--num_modes", str(N_POSES),
        "--energy_range", str(ENERGY_RANGE),
        "--cpu", str(CPU),
    ]
    subprocess.run(cmd, check=True, capture_output=True, text=True, timeout=PER_LIGAND_TIMEOUT)
    return _best_affinity_from_pose(pose_pdbqt) if pose_pdbqt.exists() else None


def _append_checkpoint(row: dict) -> None:
    pd.DataFrame([row]).to_csv(
        CHECKPOINT_FILE, mode="a", header=not CHECKPOINT_FILE.exists(), index=False
    )


# ------------------------------------------------------------
# Dock a set of candidates, with resume + checkpointing.
# ------------------------------------------------------------
def dock_candidates(candidates: pd.DataFrame) -> pd.DataFrame:
    ensure_vina_binary()

    done_cids = set()
    if RESUME and CHECKPOINT_FILE.exists():
        try:
            done_cids = set(pd.read_csv(CHECKPOINT_FILE)["cid"].astype(int))
            print(f"Resuming: {len(done_cids)} compounds already docked (skipped).")
        except Exception:
            done_cids = set()

    todo = candidates[~candidates["cid"].astype(int).isin(done_cids)].reset_index(drop=True)
    total = len(todo)
    print(f"Compounds to dock now: {total} (of {len(candidates)} selected)")

    for i, row in todo.iterrows():
        cid = int(row["cid"])
        name = str(row.get("compound_name", f"CID{cid}"))
        lig_pdbqt = LIG_DIR / f"CID{cid}.pdbqt"
        pose_file = POSE_DIR / f"CID{cid}_out.pdbqt"
        try:
            if not prepare_ligand_pdbqt(row["canonical_smiles"], lig_pdbqt):
                print(f"  [{i+1}/{total}] [skip] {name} (CID{cid}): ligand prep failed")
                _append_checkpoint({"cid": cid, "compound_name": name,
                                    "vina_best_affinity_kcal_mol": np.nan, "pose_file": ""})
                continue
            best = dock_one_ligand(lig_pdbqt, pose_file)
            if best is None:
                raise RuntimeError("no affinity parsed from pose")
            print(f"  [{i+1}/{total}] {name} (CID{cid}): {best:.2f} kcal/mol")
            _append_checkpoint({"cid": cid, "compound_name": name,
                                "vina_best_affinity_kcal_mol": round(best, 2),
                                "pose_file": str(pose_file)})
        except Exception as e:
            print(f"  [{i+1}/{total}] [error] {name} (CID{cid}): {type(e).__name__}: {str(e)[:80]}")
            _append_checkpoint({"cid": cid, "compound_name": name,
                                "vina_best_affinity_kcal_mol": np.nan, "pose_file": ""})

    return pd.read_csv(CHECKPOINT_FILE) if CHECKPOINT_FILE.exists() else pd.DataFrame()


# ------------------------------------------------------------
# Driver -- select the top-50 ACTIVE candidates, then dock.
# ------------------------------------------------------------
def _load_source_from_disk() -> tuple[pd.DataFrame, str] | None:
    """Fallback: read a saved prediction table from disk (survives Colab restarts)."""
    candidates_files = [
        ("04_fda_predictions_with_bayesian_classification.csv", "consensus (from disk)"),
        ("05_fda_recA_predictions.csv", "QSAR predictions (from disk)"),
        ("05_fda_selected_compounds.csv", "compound list (from disk)"),
    ]
    for fname, desc in candidates_files:
        fpath = _find_file(fname)
        if fpath is not None:
            try:
                df = pd.read_csv(fpath)
                if {"cid", "canonical_smiles"}.issubset(df.columns):
                    print(f"Loaded predictions from disk: {fpath}")
                    return df, desc
            except Exception:
                continue
    return None


def run_docking() -> pd.DataFrame:
    source = None
    desc = ""
    if "fda_bayesian_df" in globals() and isinstance(fda_bayesian_df, pd.DataFrame) and not fda_bayesian_df.empty:
        rank_col = "consensus_confidence_score" if "consensus_confidence_score" in fda_bayesian_df.columns \
            else "combined_qsar_bayesian_score"
        source = fda_bayesian_df.sort_values(rank_col, ascending=False)
        desc = f"consensus ranking ('{rank_col}')"
    elif "fda_prediction_df" in globals() and isinstance(fda_prediction_df, pd.DataFrame) and not fda_prediction_df.empty:
        source = fda_prediction_df.sort_values("predicted_probability_active", ascending=False)
        desc = "QSAR prediction ranking"
    elif "compounds_df" in globals() and isinstance(compounds_df, pd.DataFrame) and not compounds_df.empty:
        source = compounds_df.copy()
        desc = "full compound list (compounds_df)"
    else:
        # Not in memory (e.g. runtime restarted) -> try the saved CSVs.
        loaded = _load_source_from_disk()
        if loaded is None:
            raise ValueError(
                "No compound dataframe found in memory OR on disk. Run the FDA "
                "prediction (and Bayesian scoring) cell FIRST so it produces "
                "fda_prediction_df / fda_bayesian_df and the saved CSVs."
            )
        source, desc = loaded
        # order it if a ranking column is present
        for rc in ("consensus_confidence_score", "combined_qsar_bayesian_score",
                   "predicted_probability_active"):
            if rc in source.columns:
                source = source.sort_values(rc, ascending=False)
                break

    print(f"Source: {desc}.")

    needed = {"cid", "canonical_smiles"}
    if not needed.issubset(source.columns):
        raise ValueError(f"Compound table must contain {needed}. Found: {list(source.columns)}")

    candidates = source.drop_duplicates("cid").reset_index(drop=True)

    prob_col = "predicted_probability_active"
    if prob_col in candidates.columns:
        active = candidates[pd.to_numeric(candidates[prob_col], errors="coerce") >= MIN_ACTIVE_PROB]
        if len(active) > 0:
            print(f"Filtered to predicted-active compounds ({prob_col} >= {MIN_ACTIVE_PROB}): "
                  f"{len(active)} of {len(candidates)}.")
            candidates = active.reset_index(drop=True)
        else:
            print(f"No compound reached {prob_col} >= {MIN_ACTIVE_PROB}; docking top-ranked anyway.")

    candidates = candidates.head(int(TOP_N_DOCK)).reset_index(drop=True)
    print(f"\nDocking the TOP {len(candidates)} active candidates.")

    docking_df = dock_candidates(candidates)
    if docking_df.empty:
        print("No compounds were docked.")
        return docking_df

    docking_df = docking_df.dropna(subset=["vina_best_affinity_kcal_mol"]).copy()
    docking_df = docking_df.sort_values("vina_best_affinity_kcal_mol").reset_index(drop=True)
    docking_df.insert(0, "docking_rank", np.arange(1, len(docking_df) + 1))

    merged = candidates.merge(
        docking_df.drop(columns=[c for c in ["compound_name"] if c in docking_df.columns]),
        on="cid", how="left",
    ).sort_values("vina_best_affinity_kcal_mol").reset_index(drop=True)

    merged.to_csv(DOCKING_RESULTS_FILE, index=False)
    if "save_table" in globals():
        try:
            save_table(docking_df, "05_vina_docking_results.csv")
        except Exception:
            pass

    print(f"\nDocked {len(docking_df)} compounds. Saved -> {DOCKING_RESULTS_FILE}")
    print("\nTop 15 by docking affinity (most negative = strongest predicted binding):")
    show = [c for c in ["docking_rank", "compound_name", "cid",
                        "predicted_probability_active", "combined_qsar_bayesian_score",
                        "vina_best_affinity_kcal_mol"] if c in merged.columns]
    print(merged[show].head(15).to_string(index=False))
    return merged


docking_results_df = run_docking()
if "display" in globals():
    try:
        display(docking_results_df.head(20))
    except Exception:
        pass

##Final output summary

This concise notebook generates only the core outputs requested for review:

* 01_feature_selection_summary.csv
* 01_final_selected_features.csv
* 02_qsar_model_test_set_results.csv
* 03_fda_dataset_qsar_prediction_results.csv
* 04_bayesian_classification_holdout_validation_results.csv

The notebook is intentionally limited to the final review pathway and excludes exploratory or intermediate scripts.

In [ ]:
# ============================================================
# Final output checklist + download all results
# ============================================================

import json
import zipfile
import datetime
from pathlib import Path

# ------------------------------------------------------------
# Resolve directories safely.
# ------------------------------------------------------------
_TABLE_DIR = TABLE_DIR if "TABLE_DIR" in globals() else (OUTPUT_DIR / "tables")

# Only the outputs actually produced by THIS notebook (earlier-notebook files removed).
output_checklist = {
    "fda_qsar_predictions": str(_TABLE_DIR / "03_fda_dataset_qsar_prediction_results.csv"),
    "bayesian_holdout_validation": str(_TABLE_DIR / "04_bayesian_classification_holdout_validation_results.csv"),
    "fda_bayesian_predictions": str(_TABLE_DIR / "04_fda_predictions_with_bayesian_classification.csv"),
}

# ------------------------------------------------------------
# Verify which expected outputs actually exist.
# ------------------------------------------------------------
detailed_checklist = {}
n_present = 0
print("Output checklist")
print("-" * 60)
for key, path in output_checklist.items():
    exists = Path(path).exists()
    detailed_checklist[key] = {"path": path, "exists": exists}
    n_present += int(exists)
    print(f"[{'OK     ' if exists else 'MISSING'}] {key}")
print("-" * 60)
print(f"{n_present}/{len(output_checklist)} expected files present.")

checklist_file = OUTPUT_DIR / "concise_output_checklist.json"
checklist_file.parent.mkdir(parents=True, exist_ok=True)
with open(checklist_file, "w", encoding="utf-8") as f:
    json.dump(detailed_checklist, f, indent=2)
print("\nChecklist saved:", checklist_file)

# ============================================================
# Zip EVERYTHING under the top-level outputs folder and download it.
# This captures both the AID_2221 prediction outputs and the Bayesian
# tables/figures, which live in different sub-folders.
# ============================================================

def _find_outputs_root(p: Path) -> Path:
    """Walk up to the folder named 'outputs'; fall back to the given path."""
    p = Path(p).resolve()
    for parent in [p, *p.parents]:
        if parent.name == "outputs":
            return parent
    return p


# Gather every 'outputs' root we know about (notebook + AID script may differ).
_roots = set()
for _name in ("OUTPUT_DIR", "TABLE_DIR", "FIGURE_DIR", "MODEL_DIR"):
    if _name in globals():
        _roots.add(_find_outputs_root(globals()[_name]))
# Also include the default Colab location if present.
_cwd_outputs = Path.cwd() / "outputs"
if _cwd_outputs.exists():
    _roots.add(_cwd_outputs)

# Collapse to the shallowest common roots (avoid zipping a folder twice).
roots = sorted({r for r in _roots if r.exists()}, key=lambda p: len(str(p)))
if not roots:
    roots = [OUTPUT_DIR]

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = Path.cwd() / f"recA_qsar_all_outputs_{stamp}.zip"

file_count = 0
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    seen = set()
    for root in roots:
        base = root.parent  # keep the 'outputs/...' prefix inside the archive
        for f in sorted(root.rglob("*")):
            if f.is_file() and f.resolve() not in seen:
                seen.add(f.resolve())
                zf.write(f, f.relative_to(base))
                file_count += 1

size_mb = zip_path.stat().st_size / 1e6
print(f"\nArchived {file_count} files -> {zip_path.name} ({size_mb:.2f} MB)")

# ------------------------------------------------------------
# Trigger the browser download when running in Google Colab.
# ------------------------------------------------------------
try:
    from google.colab import files  # type: ignore
    files.download(str(zip_path))
    print("Download started in your browser.")
except Exception:
    print("Not running in Google Colab — download the archive manually from:")
    print(f"  {zip_path}")